# OpenBind EV-A71/CVA16 2A protease: five-fold results analysis

This notebook reproduces the dataset summaries, tables and figures used in the Results and Discussion chapter. It analyses the **five-fold compound-grouped random and scaffold experiments only**:

1. Morgan fingerprint MLP;
2. 2D GNN;
3. crystallographic 3D distance GNN;
4. crystallographic 3D ALIGNN;
5. adapted ligand MGT;
6. masked ALIGNN; and
7. masked adapted MGT.

All primary model comparisons use the 474 pooled out-of-fold **compound-level** predictions. The notebook does not retrain models. It reads frozen results and writes publication-ready figures and tables to `output/dissertation_analysis/`.

### Running interactively

If the `mgt` Conda environment is not yet registered as a notebook kernel:

```bash
conda activate mgt
conda install -c conda-forge jupyterlab ipykernel -y
python -m ipykernel install --user --name mgt --display-name "Python (mgt)"
jupyter lab notebooks/openbind_results_analysis.ipynb
```

## Code used: Dependency setup

I keep the existing `adjustText` installation cell before the shared imports so a top-to-bottom run can import it immediately. This only prepares plot-label placement; it does not install or train the molecular models.


In [ ]:
# Install the label-placement dependency before its centralised import below.
%pip install adjustText

## Code used: Shared imports and analysis settings

I load the libraries once, locate the saved CV outputs, and define the model labels and plotting defaults used throughout this notebook. Parameter counts are recorded explicitly here; they are not recalculated from checkpoints.

`Path`, `json` and the runtime modules handle files and the session audit. NumPy, pandas and SciPy handle arrays, tables and statistics. RDKit builds fingerprints and molecular drawings; Pillow and `BytesIO` assemble image panels. Matplotlib draws the figures, `adjustText` positions labels, and `textwrap` keeps long labels readable.


In [ ]:
# Files, reproducibility settings and the final session audit.
from pathlib import Path
from io import BytesIO
import json
import math
import platform
import sys
import warnings
import textwrap

# Compound-level tables, fingerprint arrays and evaluation statistics.
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Draw, rdFingerprintGenerator

# Figure construction, molecular-image tiles and collision-aware labels.
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from PIL import Image, ImageDraw, ImageFont
from adjustText import adjust_text

# All downstream cells reuse these imports and the same saved-run paths.
# Keep notebook display with the original plain-print fallback.
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the MGT root whether the notebook is opened from MGT/ or MGT/notebooks/.
candidate = Path.cwd().resolve()
if (candidate / "train_openbind_mgt.py").is_file():
    ROOT = candidate
elif (candidate.parent / "train_openbind_mgt.py").is_file():
    ROOT = candidate.parent
else:
    matches = [parent for parent in candidate.parents if (parent / "train_openbind_mgt.py").is_file()]
    if not matches:
        raise FileNotFoundError("Run this notebook from the MGT repository or its notebooks directory.")
    ROOT = matches[0]

DATA_ROOT = ROOT / "OpenBind_EV-A71_2A" / "experiment_a_ligand_mgt"
RESULT_ROOTS = {
    "random": ROOT / "output" / "openbind_random_cv",
    "scaffold": ROOT / "output" / "openbind_scaffold_cv",
}
ANALYSIS_ROOT = ROOT / "output" / "dissertation_analysis"
FIGURE_ROOT = ANALYSIS_ROOT / "figures"
TABLE_ROOT = ANALYSIS_ROOT / "tables"
LIGAND_ROOT = FIGURE_ROOT / "ligand_error_panels"
for directory in (ANALYSIS_ROOT, FIGURE_ROOT, TABLE_ROOT, LIGAND_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

# Consistent dissertation-facing model order, labels, colours and parameter counts.
MODEL_ORDER = [
    "morgan_mlp", "2d_gnn", "3d_gnn", "3d_alignn",
    "adapted_mgt", "masked_alignn", "masked_mgt",
]
MODEL_LABELS = {
    "morgan_mlp": "Morgan MLP",
    "2d_gnn": "2D GNN",
    "3d_gnn": "3D distance GNN",
    "3d_alignn": "3D ALIGNN",
    "adapted_mgt": "Adapted MGT",
    "masked_alignn": "Masked ALIGNN",
    "masked_mgt": "Masked MGT",
}
MODEL_COLOURS = {
    "morgan_mlp": "#7f7f7f",
    "2d_gnn": "#4c78a8",
    "3d_gnn": "#59a14f",
    "3d_alignn": "#f28e2b",
    "adapted_mgt": "#b07aa1",
    "masked_alignn": "#e15759",
    "masked_mgt": "#9c755f",
}
# These are recorded model sizes, not a fresh count of checkpoint tensors.
PARAMETER_COUNTS = {
    "morgan_mlp": 270_977,
    "2d_gnn": 4_094_849,
    "3d_gnn": 4_099_969,
    "3d_alignn": 8_118_529,
    "adapted_mgt": 13_702_241,
    "masked_alignn": 8_118_529,
    "masked_mgt": 13_702_241,
}

# Plot settings favour legible dissertation figures and vector-safe fonts.
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
RNG_SEED = 123
# Fix the number of paired resamples so repeat runs use the same procedure.
N_BOOTSTRAP = 5_000

print(f"Repository: {ROOT}")
print(f"Analysis outputs: {ANALYSIS_ROOT}")

## 1. Load and audit the frozen data

This section verifies the dataset size, compound grouping, fold coverage and availability of all out-of-fold predictions before any figure is produced. Assertions intentionally stop the notebook if the frozen artifacts no longer match the completed experiment.

### Code used: Input audit

I load the curated records, fold metadata and saved predictions, then check compound coverage and all five outer folds for each model. Compound IDs are padded consistently before matching files.


In [ ]:
# Require complete compound-level out-of-fold files before comparing models.
# Read the saved metadata without regenerating dataset or CV artifacts.
def read_json(path: Path) -> dict:
    """Read one UTF-8 JSON file."""
    return json.loads(path.read_text(encoding="utf-8"))


# Use the runner's compound-level out-of-fold files, not per-structure predictions.
def prediction_path(method: str, model: str) -> Path:
    """Return the pooled compound prediction file for one method/model."""
    return RESULT_ROOTS[method] / "summary" / f"{model}_out_of_fold_compound_predictions.csv"


dataset_metadata = read_json(DATA_ROOT / "reports" / "dataset_metadata.json")
curated_structures = pd.read_csv(DATA_ROOT / "curated" / "openbind_ligand_structures.csv")
curated_compounds = pd.read_csv(DATA_ROOT / "curated" / "openbind_compounds.csv")
excluded_records = pd.read_csv(DATA_ROOT / "curated" / "excluded_records.csv")

cv_metadata = {
    "random": read_json(DATA_ROOT / "cv_random" / "cv_metadata.json"),
    "scaffold": read_json(DATA_ROOT / "cv" / "cv_metadata.json"),
}
cv_summaries = {
    method: read_json(root / "summary" / "cv_summary.json")
    for method, root in RESULT_ROOTS.items()
}
fold_metrics = {
    method: pd.read_csv(root / "summary" / "fold_metrics.csv")
    for method, root in RESULT_ROOTS.items()
}

assert len(curated_structures) == 621, "Expected 621 curated structure rows."
assert len(curated_compounds) == 474, "Expected 474 curated compound groups."
assert curated_structures["complex_name"].is_unique, "Structure names must be unique."
assert curated_compounds["official_compound_group_id"].astype(str).nunique() == 474

# Preserve leading zeroes in the 16-character compound hash identifiers.
curated_compounds["official_compound_group_id"] = curated_compounds[
    "official_compound_group_id"
].astype(str).str.zfill(16)

# Audit every model under both grouping designs before downstream plots.
for method in ("random", "scaffold"):
    assert cv_metadata[method]["total_compounds"] == 474
    assert cv_metadata[method]["total_structures"] == 621
    for model in MODEL_ORDER:
        predictions = pd.read_csv(
            prediction_path(method, model),
            dtype={"official_compound_group_id": str},
        )
        predictions["official_compound_group_id"] = predictions[
            "official_compound_group_id"
        ].str.zfill(16)
        assert len(predictions) == 474, f"Incomplete {method}/{model} prediction file."
        assert predictions["official_compound_group_id"].is_unique
        assert set(predictions["outer_fold"]) == set(range(5))

print("Audit passed: 621 structures, 474 compounds, 70 completed fold fits and complete OOF predictions.")

## 2. Dataset curation and affinity distribution

The first table follows the dataset from the OpenBind release to the final modelling cohort. Exclusion-reason counts are shown separately because one record can satisfy more than one exclusion criterion.

### Code used: Curation summary

I combine the recorded preparation counts with the explicitly listed release/reference cohort sizes. Exclusion reasons are tabulated separately because one structure can have more than one reason.


In [ ]:
# Keep structure counts, compound counts and overlapping exclusion reasons distinct.
counts = dataset_metadata["counts"]
curation_table = pd.DataFrame([
    {"Dataset stage": "OpenBind release", "Structures": counts["source_metadata_rows"], "Compounds": 699},
    {"Dataset stage": "Release compounds with affinity", "Structures": np.nan, "Compounds": 601},
    {"Dataset stage": "Official benchmark affinity reference", "Structures": np.nan, "Compounds": 494},
    {"Dataset stage": "Final curated modelling cohort", "Structures": counts["curated_structure_rows"], "Compounds": counts["benchmark_compound_groups"]},
    {"Dataset stage": "Excluded structure records", "Structures": counts["excluded_structure_rows"], "Compounds": np.nan},
])
curation_table.to_csv(TABLE_ROOT / "table_dataset_curation.csv", index=False)

# Exclusion categories can overlap; their counts need not sum to excluded structures.
exclusion_table = (
    pd.Series(dataset_metadata["exclusion_reason_counts"], name="Record count")
    .rename_axis("Exclusion reason")
    .reset_index()
    .sort_values("Record count", ascending=False)
)
exclusion_table.to_csv(TABLE_ROOT / "table_exclusion_reasons.csv", index=False)

display(curation_table)
display(exclusion_table)

### Code used: Compound-affinity distribution

I summarise pKD from the compound table, not the repeated structure rows, and export the histogram with its underlying summary. The standard deviation uses the population convention, `ddof=0`.


In [ ]:
# Use one affinity label per compound, regardless of its number of crystal structures.
pkd = curated_compounds["experimental_pKD"].to_numpy(float)
# Population SD describes the full curated compound cohort here.
pkd_summary = pd.DataFrame({
    "Statistic": ["Compounds", "Minimum", "Maximum", "Mean", "Median", "Population SD"],
    "Value": [len(pkd), pkd.min(), pkd.max(), pkd.mean(), np.median(pkd), pkd.std(ddof=0)],
})
pkd_summary.to_csv(TABLE_ROOT / "table_pkd_summary.csv", index=False)
display(pkd_summary.round(4))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
bins = np.linspace(math.floor(pkd.min() * 2) / 2, math.ceil(pkd.max() * 2) / 2, 19)
ax.hist(pkd, bins=bins, color="#4c78a8", edgecolor="white", alpha=0.9)
ax.axvline(pkd.mean(), color="#e15759", linestyle="--", linewidth=2, label=f"Mean = {pkd.mean():.2f}")
ax.axvline(np.median(pkd), color="#f28e2b", linestyle=":", linewidth=2, label=f"Median = {np.median(pkd):.2f}")
ax.set(xlabel="Experimental pKD", ylabel="Number of compounds", title="OpenBind curated compound-affinity distribution")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_pkd_distribution.png", bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "figure_pkd_distribution.pdf", bbox_inches="tight")
plt.show()

### Code used: Repeated structures

I count how many compounds have each number of crystallographic structures. This describes observation multiplicity without treating those structures as independent compound labels.


In [ ]:
# Count repeated observations without duplicating compounds in the affinity summary.
structure_counts = curated_compounds["structure_count"].astype(int)
multiplicity = structure_counts.value_counts().sort_index().rename_axis("Structures per compound").reset_index(name="Compounds")
multiplicity.to_csv(TABLE_ROOT / "table_structure_multiplicity.csv", index=False)
display(multiplicity)

fig, ax = plt.subplots(figsize=(6.5, 4.1))
ax.bar(multiplicity["Structures per compound"].astype(str), multiplicity["Compounds"], color="#59a14f")
for x, value in enumerate(multiplicity["Compounds"]):
    ax.text(x, value + 4, str(value), ha="center", va="bottom", fontsize=9)
ax.set(xlabel="Crystallographic structures per compound", ylabel="Number of compounds", title="Repeated crystallographic observations")
ax.set_ylim(0, multiplicity["Compounds"].max() * 1.13)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_structure_multiplicity.png", bbox_inches="tight")
plt.show()

## 3. Fingerprint chemical-space and fold analysis

Morgan fingerprints are projected using a deterministic two-component principal-coordinate representation derived from the centred fingerprint Gram matrix. This is a visual diagnostic, not a supervised model result. Pairwise Tanimoto distributions quantify chemical similarity within and between outer folds.

### Code used: Fingerprint projection and fold similarity

I create radius-2 Morgan fingerprints, centre their bit matrix and obtain two projection components from its Gram-matrix eigendecomposition. The explained-variance percentages come from the retained eigenvalues divided by their total, not from the model scores. I then compare unique compound-pair Tanimoto values within and between outer folds.


In [ ]:
# Use one fixed compound order for both the fingerprint arrays and fold labels.
assignment_paths = {
    "random": DATA_ROOT / "cv_random" / "random_grouped_5fold_seed_123.csv",
    "scaffold": DATA_ROOT / "cv" / "scaffold_grouped_5fold_seed_123.csv",
}
assignments = {
    method: pd.read_csv(path, dtype={"official_compound_group_id": str})
    for method, path in assignment_paths.items()
}
for frame in assignments.values():
    frame["official_compound_group_id"] = frame["official_compound_group_id"].str.zfill(16)

base = assignments["random"].sort_values("official_compound_group_id").reset_index(drop=True)
assert set(base["official_compound_group_id"]) == set(assignments["scaffold"]["official_compound_group_id"])

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = []
fingerprint_array = []
valid_ids = []
for row in base.itertuples(index=False):
    molecule = Chem.MolFromSmiles(row.canonical_smiles)
    if molecule is None:
        raise ValueError(f"Could not parse {row.official_compound_group_id}")
    fingerprint = fingerprint_generator.GetFingerprint(molecule)
    vector = np.zeros(2048, dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fingerprint, vector)
    fingerprints.append(fingerprint)
    fingerprint_array.append(vector)
    valid_ids.append(row.official_compound_group_id)
X = np.asarray(fingerprint_array, dtype=np.float32)

# Two-dimensional scores from the centred Gram matrix avoid a scikit-learn dependency.
# Centre fingerprint bits; this projection is independent of affinity model predictions.
X_centered = X - X.mean(axis=0, keepdims=True)
# Recover PCA scores through the smaller compound-space Gram matrix.
gram = X_centered @ X_centered.T
eigenvalues, eigenvectors = np.linalg.eigh(gram)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = np.maximum(eigenvalues[order], 0)
scores = eigenvectors[:, order[:2]] * np.sqrt(eigenvalues[:2])
# The two percentages are eigenvalue shares; their sum gives the reported two-axis coverage.
explained = eigenvalues[:2] / eigenvalues.sum()

# Vectorised pairwise Tanimoto similarity for the binary fingerprint matrix.
# For binary fingerprints, dot products count shared on-bits.
intersection = X @ X.T
bit_counts = X.sum(axis=1)
union = bit_counts[:, None] + bit_counts[None, :] - intersection
tanimoto = np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)
# Exclude self-pairs and count each unordered compound pair once.
upper = np.triu_indices(len(X), k=1)

fold_colours = ["#4c78a8", "#f28e2b", "#59a14f", "#e15759", "#b07aa1"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
similarity_rows = []
for column, method in enumerate(("random", "scaffold")):
    fold_map = assignments[method].set_index("official_compound_group_id")["outer_fold"]
    folds = np.asarray([int(fold_map.loc[compound_id]) for compound_id in valid_ids])
    ax = axes[0, column]
    for fold in range(5):
        mask = folds == fold
        ax.scatter(scores[mask, 0], scores[mask, 1], s=22, alpha=0.72, color=fold_colours[fold], label=f"Fold {fold}")
    ax.set(
        xlabel=f"Fingerprint component 1 ({explained[0] * 100:.1f}% variance)",
        ylabel=f"Fingerprint component 2 ({explained[1] * 100:.1f}% variance)",
        title=f"{method.capitalize()} outer-fold assignment",
    )
    ax.legend(frameon=False, ncol=2)

    # Apply each CV assignment to the same pairwise similarity matrix.
    same_fold = folds[upper[0]] == folds[upper[1]]
    within = tanimoto[upper][same_fold]
    between = tanimoto[upper][~same_fold]
    similarity_rows.extend([
        {"CV method": method, "Comparison": "Within outer fold", "Mean Tanimoto": within.mean(), "Median Tanimoto": np.median(within), "Pairs": len(within)},
        {"CV method": method, "Comparison": "Between outer folds", "Mean Tanimoto": between.mean(), "Median Tanimoto": np.median(between), "Pairs": len(between)},
    ])
    axes[1, column].hist(
    within,
    bins=np.linspace(0, 1, 31),
    density=True,
    alpha=0.25,
    color="#4c78a8",
    edgecolor="#4c78a8",
    linewidth=0.6,
    label="Within fold",
    )

    axes[1, column].hist(
    between,
    bins=np.linspace(0, 1, 31),
    density=True,
    alpha=0.25,
    color="#f28e2b",
    edgecolor="#f28e2b",
    linewidth=0.6,
    label="Between folds",
    )
    axes[1, column].set(xlabel="Pairwise Morgan Tanimoto similarity", ylabel="Density", title=f"{method.capitalize()} fold similarity")
    axes[1, column].legend(frameon=False)

fig.suptitle("Chemical-space distribution under random and scaffold cross-validation", y=1.01, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "figure_chemical_space_folds.png", bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "figure_chemical_space_folds.pdf", bbox_inches="tight")
plt.show()

similarity_table = pd.DataFrame(similarity_rows)
similarity_table.to_csv(TABLE_ROOT / "table_fold_tanimoto_similarity.csv", index=False)
display(similarity_table.round(4))

## 4. Five-fold model-performance tables

`Pooled` metrics are calculated from all 474 non-overlapping out-of-fold compound predictions. `Fold mean ± SD` describes variation across the five outer folds. These are different summaries and must not be conflated.

### Code used: CV performance tables

I read the pooled metrics and fold mean/SD from the runner summaries and combine them with the model labels. Each CV design gets a separate CSV; displayed rounding does not alter the saved values.


In [ ]:
# Preserve the distinction between pooled compound metrics and variation across folds.
# Take pooled metrics from the full OOF summary and retain fold variation separately.
def build_performance_table(method: str) -> pd.DataFrame:
    """Combine pooled metrics and fold variation for one CV design."""
    pooled = cv_summaries[method]["pooled_out_of_fold_metrics"]
    rows = []
    for model in MODEL_ORDER:
        metric = pooled[model]
        rows.append({
            "model_key": model,
            "Model": MODEL_LABELS[model],
            "Parameters": PARAMETER_COUNTS[model],
            "Fold RMSE mean": metric["fold_mean_rmse"],
            "Fold RMSE SD": metric["fold_sd_rmse"],
            "Pooled MAE": metric["mae"],
            "Pooled RMSE": metric["rmse"],
            "Pooled R2": metric["r2"],
            "Pooled Pearson": metric["pearson_r"],
            "Pooled Spearman": metric["spearman_r"],
        })
    return pd.DataFrame(rows)


performance = {method: build_performance_table(method) for method in ("random", "scaffold")}
for method, table in performance.items():
    table.to_csv(TABLE_ROOT / f"table_{method}_cv_performance.csv", index=False)
    print(f"\n{method.upper()} FIVE-FOLD CV")
    display(table.drop(columns="model_key").round(4))

### Code used: Fold-RMSE plot

I plot the outer-fold scores alongside their mean and spread, keeping the two CV designs separate. This describes fold variation rather than a confidence interval for the pooled metric.


In [ ]:
# Plot fold-level variation without treating fold SD as a confidence interval.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True)
for ax, method in zip(axes, ("random", "scaffold")):
    fold_frame = fold_metrics[method].copy()
    y_positions = np.arange(len(MODEL_ORDER))
    for y, model in zip(y_positions, MODEL_ORDER):
        values = fold_frame.loc[fold_frame["model_key"] == model, "rmse"].to_numpy(float)
        jitter = np.linspace(-0.10, 0.10, len(values))
        ax.scatter(values, y + jitter, s=28, alpha=0.65, color=MODEL_COLOURS[model], zorder=2)
        row = performance[method].set_index("model_key").loc[model]
        ax.errorbar(
            row["Fold RMSE mean"], y,
            xerr=row["Fold RMSE SD"],
            fmt="D", markersize=6, color="black", ecolor="black", capsize=3, zorder=3,
        )
    ax.set_yticks(y_positions, [MODEL_LABELS[model] for model in MODEL_ORDER])
    ax.invert_yaxis()
    ax.set(xlabel="Compound-level RMSE (pKD)", title=f"{method.capitalize()} five-fold CV")
    ax.grid(axis="x", alpha=0.25)
axes[0].set_ylabel("Model")
legend = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#777777", alpha=0.7, label="Outer-fold RMSE"),
    Line2D([0], [0], marker="D", color="black", linestyle="none", label="Fold mean ± fold SD"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Model performance across random and scaffold cross-validation", fontsize=14)
fig.tight_layout(rect=(0, 0.06, 1, 0.95))
fig.savefig(FIGURE_ROOT / "figure_cv_model_rmse.png", bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "figure_cv_model_rmse.pdf", bbox_inches="tight")
plt.show()

### Code used: Random-versus-scaffold differences

I join the two performance tables by model key and subtract random pooled RMSE from scaffold pooled RMSE. The zero line gives a common reference for the signed differences.


In [ ]:
# Positive differences mean higher pooled error under scaffold CV.
# Compare each model's pooled scaffold error with its pooled random error.
comparison = (
    performance["random"][
        ["model_key", "Model", "Pooled RMSE"]
    ]
    .rename(columns={"Pooled RMSE": "Random RMSE"})
)

# Add the corresponding scaffold-CV RMSE.
comparison = comparison.merge(
    performance["scaffold"][
        ["model_key", "Pooled RMSE"]
    ].rename(columns={"Pooled RMSE": "Scaffold RMSE"}),
    on="model_key",
)

# Calculate the change in error from random to scaffold CV.
comparison["Scaffold minus random RMSE"] = (
    comparison["Scaffold RMSE"]
    - comparison["Random RMSE"]
)

# Save and display the numerical results.
comparison.to_csv(
    TABLE_ROOT / "table_random_vs_scaffold.csv",
    index=False,
)

display(
    comparison
    .drop(columns="model_key")
    .round(4)
)

# Create the bar chart.
fig, ax = plt.subplots(figsize=(8, 4.8))

x = np.arange(len(comparison))
values = comparison["Scaffold minus random RMSE"].to_numpy()

# Draw the bars.
bars = ax.bar(
    x,
    values,
    color=[
        MODEL_COLOURS[key]
        for key in comparison["model_key"]
    ],
    zorder=2,
)

# Draw the zero-reference line above the bars.
ax.axhline(
    0,
    color="black",
    linewidth=1.1,
    zorder=3,
)

# Calculate y-axis padding so negative labels do not touch the x-axis.
value_range = values.max() - values.min()
padding = max(0.004, value_range * 0.15)

lower_limit = min(0, values.min()) - padding
upper_limit = max(0, values.max()) + padding

ax.set_ylim(lower_limit, upper_limit)

# Configure the model labels.
ax.set_xticks(
    x,
    comparison["Model"],
    rotation=35,
    ha="right",
)

# Add the axis label and title.
ax.set(
    ylabel="Scaffold RMSE − random RMSE (pKD)",
    title="Change in pooled error under scaffold cross-validation",
)

# Add light horizontal grid lines.
ax.grid(
    axis="y",
    alpha=0.20,
    linewidth=0.7,
    zorder=0,
)

# Add values above positive bars and below negative bars.
for bar, value in zip(bars, values):

    # Positive labels move upward; negative labels move downward.
    vertical_offset = 6 if value >= 0 else -7
    vertical_alignment = "bottom" if value >= 0 else "top"

    ax.annotate(
        f"{value:+.3f}",
        xy=(
            bar.get_x() + bar.get_width() / 2,
            value,
        ),
        xytext=(0, vertical_offset),
        textcoords="offset points",
        ha="center",
        va=vertical_alignment,
        fontsize=8,
    )

# Remove unnecessary upper and right borders.
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Prevent labels from being cropped.
fig.tight_layout()

# Save the revised figure.
fig.savefig(
    FIGURE_ROOT / "figure_random_scaffold_delta.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Masked versus unmasked models

The sign convention below is `masked RMSE − unmasked RMSE`; negative values indicate improvement after pretraining.

### Code used: Masked-versus-unmasked differences

I pair each masked model with its unmasked architecture within the same CV design and calculate masked RMSE minus unmasked RMSE. This cell analyses saved predictions; it does not run pretraining.


In [ ]:
# Pair masked and unmasked versions within each CV design before subtracting RMSE.
# Collect the effect of masking for ALIGNN and adapted MGT.
masking_rows = []

# Evaluate masking separately under random and scaffold CV.
for method in ("random", "scaffold"):

    # Index the performance table by model identifier.
    indexed = performance[method].set_index("model_key")

    # Compare each masked model with its corresponding unmasked model.
    for architecture, unmasked, masked in [
        ("ALIGNN", "3d_alignn", "masked_alignn"),
        ("Adapted MGT", "adapted_mgt", "masked_mgt"),
    ]:

        # Obtain the pooled compound-level RMSE values.
        unmasked_rmse = indexed.loc[unmasked, "Pooled RMSE"]
        masked_rmse = indexed.loc[masked, "Pooled RMSE"]

        # Store the absolute results and masking-induced change.
        masking_rows.append(
            {
                "CV method": method,
                "Architecture": architecture,
                "Unmasked RMSE": unmasked_rmse,
                "Masked RMSE": masked_rmse,
                "Masked minus unmasked RMSE": (
                    masked_rmse - unmasked_rmse
                ),
            }
        )

# Create and save the masking comparison table.
masking_table = pd.DataFrame(masking_rows)

masking_table.to_csv(
    TABLE_ROOT / "table_masking_effect.csv",
    index=False,
)

display(masking_table.round(4))

# -------------------------------------------------------------------------
# Plot the change in RMSE caused by masked atom-feature pretraining
# -------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.2, 4.8))

# Define the centre of each architecture group.
positions = np.arange(2)

# Define the width of the individual bars.
width = 0.34

# Plot random and scaffold CV as adjacent bars.
for offset, method, colour in [
    (-width / 2, "random", "#4c78a8"),
    (width / 2, "scaffold", "#f28e2b"),
]:

    # Select masking effects for the current CV method.
    method_values = masking_table.loc[
        masking_table["CV method"] == method,
        "Masked minus unmasked RMSE",
    ].to_numpy()

    # Draw the bars.
    bars = ax.bar(
        positions + offset,
        method_values,
        width=width,
        color=colour,
        label=method.capitalize(),
        zorder=2,
    )

    # Add the numerical value to every bar.
    for bar, value in zip(bars, method_values):

        # Place positive labels above and negative labels below the bars.
        vertical_offset = 6 if value >= 0 else -7
        vertical_alignment = "bottom" if value >= 0 else "top"

        ax.annotate(
            f"{value:+.3f}",
            xy=(
                bar.get_x() + bar.get_width() / 2,
                value,
            ),
            xytext=(0, vertical_offset),
            textcoords="offset points",
            ha="center",
            va=vertical_alignment,
            fontsize=9,
        )

# Draw the zero-change reference line.
ax.axhline(
    0,
    color="black",
    linewidth=1.1,
    zorder=3,
)

# Obtain all plotted values to calculate suitable axis limits.
all_values = masking_table[
    "Masked minus unmasked RMSE"
].to_numpy()

# Add enough space above and below the bars and labels.
value_range = all_values.max() - all_values.min()
padding = max(0.004, value_range * 0.20)

lower_limit = min(0, all_values.min()) - padding
upper_limit = max(0, all_values.max()) + padding

ax.set_ylim(lower_limit, upper_limit)

# Label the two model architectures.
ax.set_xticks(
    positions,
    ["ALIGNN", "Adapted MGT"],
)

# Add the axis label and title.
ax.set(
    ylabel="Masked − unmasked RMSE (pKD)",
    title="Effect of masked atom-feature pretraining",
)

# Add a light horizontal reference grid.
ax.grid(
    axis="y",
    alpha=0.20,
    linewidth=0.7,
    zorder=0,
)

# Identify random and scaffold CV bars.
ax.legend(
    frameon=False,
    title="Cross-validation",
)

# Remove unnecessary borders.
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Prevent labels from being cropped.
fig.tight_layout()

# Save raster and vector versions.
fig.savefig(
    FIGURE_ROOT / "figure_masking_effect.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_ROOT / "figure_masking_effect.pdf",
    bbox_inches="tight",
)

plt.show()

## 6. Parameter efficiency

Masked models share the same supervised parameter count as their unmasked counterparts because the temporary reconstruction decoder is discarded before affinity fine-tuning.

### Code used: Parameter-efficiency plot

I pair the recorded parameter counts with pooled RMSE and draw both CV designs on comparable axes. The collision-aware helper places labels and short leader lines without changing the plotted values.


In [ ]:
# Reposition labels only; parameter counts and pooled errors remain unchanged.
# -------------------------------------------------------------------------
# Helper: route short connector lines around labels, dots and existing lines
# -------------------------------------------------------------------------

# Find short readable label connections while leaving data-point coordinates fixed.
def add_short_collision_aware_leaders(
    ax,
    labels,
    target_x,
    target_y,
    point_clearance=15,
):
    """
    Connect adjusted text labels to their corresponding data points.

    The function tests straight and curved routes and chooses the route that:
    1. avoids other model markers;
    2. avoids other text labels;
    3. avoids previously selected connector lines;
    4. remains inside the plotting area;
    5. uses the least curvature possible.
    """

    # Draw the figure so text extents can be measured.
    ax.figure.canvas.draw()
    renderer = ax.figure.canvas.get_renderer()

    # Combine target coordinates into an N × 2 array.
    point_data = np.column_stack(
        [
            np.asarray(target_x, dtype=float),
            np.asarray(target_y, dtype=float),
        ]
    )

    # Convert target coordinates into pixel/display coordinates.
    point_display = ax.transData.transform(point_data)

    # Measure every adjusted text label.
    text_boxes = [
        text.get_window_extent(renderer).expanded(1.06, 1.12)
        for text in labels
    ]

    # Obtain the visible pixel boundary of the subplot.
    axes_box = ax.get_window_extent(renderer)

    # Straight lines are tested first.
    # Small curvatures are preferred over large curvatures.
    candidate_curvatures = [
        0.0,
        0.06,
        -0.06,
        0.12,
        -0.12,
        0.20,
        -0.20,
        0.30,
        -0.30,
    ]

    # Store selected connector paths.
    accepted_paths = []

    # Store connector information.
    connector_records = []

    for index, (text, target) in enumerate(
        zip(labels, point_display)
    ):
        # Text positions remain in data coordinates after adjustText.
        text_data_position = text.get_position()

        # Convert the text position to display coordinates.
        text_display_position = ax.transData.transform(
            text_data_position
        )

        # Measure the direct pixel distance to the target.
        connector_length = np.linalg.norm(
            target - text_display_position
        )

        connector_records.append(
            {
                "index": index,
                "text": text,
                "text_data": text_data_position,
                "text_display": text_display_position,
                "target_data": point_data[index],
                "target_display": target,
                "length": connector_length,
            }
        )

    # Route longer connectors first because they have more opportunities
    # to intersect other objects.
    connector_records.sort(
        key=lambda record: record["length"],
        reverse=True,
    )

    def create_quadratic_curve(
        start,
        end,
        curvature,
        samples=120,
    ):
        """Approximate Matplotlib's arc3 connector in display coordinates."""

        start = np.asarray(start, dtype=float)
        end = np.asarray(end, dtype=float)

        difference = end - start
        midpoint = (start + end) / 2

        # Move the curve control point perpendicular to the direct path.
        control = midpoint + curvature * np.array(
            [
                difference[1],
                -difference[0],
            ]
        )

        interpolation = np.linspace(
            0,
            1,
            samples,
        )[:, None]

        curve = (
            ((1 - interpolation) ** 2) * start
            + 2
            * (1 - interpolation)
            * interpolation
            * control
            + (interpolation**2) * end
        )

        return curve

    def score_candidate_route(
        curve,
        current_index,
        curvature,
    ):
        """
        Assign a collision penalty to a candidate connector.

        Lower scores indicate shorter and clearer routes.
        """

        # Strongly prefer straight or only slightly curved connectors.
        score = abs(curvature) * 250

        # Penalise routes leaving the visible axes.
        outside_axes = (
            (curve[:, 0] < axes_box.x0 + 2)
            | (curve[:, 0] > axes_box.x1 - 2)
            | (curve[:, 1] < axes_box.y0 + 2)
            | (curve[:, 1] > axes_box.y1 - 2)
        )

        score += outside_axes.sum() * 3000

        # Penalise routes passing near another model marker.
        for point_index, point in enumerate(point_display):

            # The line must end at its own marker.
            if point_index == current_index:
                continue

            distances = np.linalg.norm(
                curve - point,
                axis=1,
            )

            minimum_distance = distances.min()

            if minimum_distance < point_clearance:
                score += (
                    point_clearance - minimum_distance
                ) * 3500

        # Penalise routes passing through another label.
        for text_index, text_box in enumerate(text_boxes):

            # The line begins inside its own text label.
            if text_index == current_index:
                continue

            intersects_text = (
                (curve[:, 0] >= text_box.x0 - 3)
                & (curve[:, 0] <= text_box.x1 + 3)
                & (curve[:, 1] >= text_box.y0 - 3)
                & (curve[:, 1] <= text_box.y1 + 3)
            )

            score += intersects_text.sum() * 5000

        # Penalise routes approaching previously accepted connectors.
        for previous_curve in accepted_paths:

            current_sample = curve[::4]
            previous_sample = previous_curve[::4]

            pairwise_distances = np.linalg.norm(
                current_sample[:, None, :]
                - previous_sample[None, :, :],
                axis=2,
            )

            minimum_line_distance = pairwise_distances.min()

            if minimum_line_distance < 5:
                score += (
                    5 - minimum_line_distance
                ) * 3000

        return score

    # Route each connector.
    for record in connector_records:

        current_index = record["index"]

        best_curve = None
        best_curvature = 0.0
        best_score = np.inf

        # Test the straight route and progressively curved alternatives.
        for curvature in candidate_curvatures:

            candidate_curve = create_quadratic_curve(
                record["text_display"],
                record["target_display"],
                curvature,
            )

            candidate_score = score_candidate_route(
                candidate_curve,
                current_index,
                curvature,
            )

            if candidate_score < best_score:
                best_curve = candidate_curve
                best_curvature = curvature
                best_score = candidate_score

        # Store the selected route.
        accepted_paths.append(best_curve)

        text = record["text"]

        # Add the selected connector.
        connector = FancyArrowPatch(
            posA=record["text_data"],
            posB=record["target_data"],

            # Clip the line at the edge of the text box.
            patchA=text.get_bbox_patch(),

            transform=ax.transData,
            connectionstyle=f"arc3,rad={best_curvature}",
            arrowstyle="-",
            color="#666666",
            linewidth=0.55,
            alpha=0.80,

            # Keep the line away from the text and marker edges.
            shrinkA=2,
            shrinkB=8,
            zorder=2,
        )

        ax.add_patch(connector)


# -------------------------------------------------------------------------
# Create and save the parameter-count table
# -------------------------------------------------------------------------

# Use the recorded trainable sizes; masking decoders are not part of these supervised counts.
parameter_table = pd.DataFrame(
    [
        {
            "model_key": model,
            "Model": MODEL_LABELS[model],
            "Trainable parameters": PARAMETER_COUNTS[model],
        }
        for model in MODEL_ORDER
    ]
)

parameter_table.to_csv(
    TABLE_ROOT / "table_parameter_counts.csv",
    index=False,
)

display(
    parameter_table.drop(columns="model_key")
)


# -------------------------------------------------------------------------
# Establish common axis limits
# -------------------------------------------------------------------------

all_rmse_values = np.concatenate(
    [
        performance[method]["Pooled RMSE"].to_numpy()
        for method in ("random", "scaffold")
    ]
)

common_y_limits = (
    all_rmse_values.min() - 0.018,
    all_rmse_values.max() + 0.016,
)

minimum_parameters = parameter_table[
    "Trainable parameters"
].min()

maximum_parameters = parameter_table[
    "Trainable parameters"
].max()


# -------------------------------------------------------------------------
# Create the figure
# -------------------------------------------------------------------------

top_two_rows = []
adjustment_jobs = []

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
    sharey=False,
)

for ax, method in zip(
    axes,
    ("random", "scaffold"),
):

    # Index the current results using the model identifier.
    indexed = performance[method].set_index("model_key")

    # Construct the plotting table.
    plot_data = pd.DataFrame(
        {
            "model_key": MODEL_ORDER,
            "Model": [
                MODEL_LABELS[model]
                for model in MODEL_ORDER
            ],
            "Parameters": [
                PARAMETER_COUNTS[model]
                for model in MODEL_ORDER
            ],
            "RMSE": [
                indexed.loc[model, "Pooled RMSE"]
                for model in MODEL_ORDER
            ],
        }
    )

    # Identify the two models with the lowest pooled RMSE.
    top_two = (
        plot_data
        .nsmallest(2, "RMSE")
        .reset_index(drop=True)
    )

    top_two["Rank"] = [1, 2]

    for _, row in top_two.iterrows():
        top_two_rows.append(
            {
                "CV method": method,
                "Rank": int(row["Rank"]),
                "model_key": row["model_key"],
                "Model": row["Model"],
                "Trainable parameters": int(row["Parameters"]),
                "Pooled RMSE": row["RMSE"],
            }
        )

    point_x = plot_data["Parameters"].to_numpy()
    point_y = plot_data["RMSE"].to_numpy()

    # ---------------------------------------------------------------------
    # Plot all model points
    # ---------------------------------------------------------------------

    for _, row in plot_data.iterrows():

        model_key = row["model_key"]

        ax.scatter(
            row["Parameters"],
            row["RMSE"],
            s=90,
            color=MODEL_COLOURS[model_key],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.82,
            zorder=3,
        )

    # Create modest invisible exclusion zones around every marker.
    # These are large enough to prevent overlap without pushing labels far away.
    point_exclusion_zones = ax.scatter(
        point_x,
        point_y,
        s=400,
        facecolors="none",
        edgecolors="none",
        alpha=0,
        zorder=2,
    )

    # ---------------------------------------------------------------------
    # Highlight the top two models
    # ---------------------------------------------------------------------

    best_model = top_two.iloc[0]
    second_model = top_two.iloc[1]

    ax.scatter(
        best_model["Parameters"],
        best_model["RMSE"],
        s=220,
        facecolors="none",
        edgecolors="#d4a017",
        linewidths=2.7,
        zorder=4,
    )

    ax.scatter(
        second_model["Parameters"],
        second_model["RMSE"],
        s=220,
        facecolors="none",
        edgecolors="#8c8c8c",
        linewidths=2.7,
        zorder=4,
    )

    # ---------------------------------------------------------------------
    # Format both axes
    # ---------------------------------------------------------------------

    ax.set_xscale("log")

    ax.set_xlim(
        minimum_parameters * 0.68,
        maximum_parameters * 1.85,
    )

    ax.set_ylim(common_y_limits)

    ax.set_xlabel(
        "Trainable parameters (log scale)",
        labelpad=9,
    )

    ax.set_ylabel(
        "Pooled compound-level RMSE (pKD)",
        labelpad=9,
    )

    ax.set_title(
        f"{method.capitalize()} five-fold CV",
        pad=14,
    )

    ax.grid(
        alpha=0.18,
        linewidth=0.7,
        zorder=0,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # ---------------------------------------------------------------------
    # Create labels close to their corresponding dots
    # ---------------------------------------------------------------------

    label_objects = []

    for row_number, (_, row) in enumerate(
        plot_data.iterrows()
    ):

        wrapped_name = textwrap.fill(
            str(row["Model"]),
            width=14,
        )

        # Alternate the initial horizontal side.
        direction = -1 if row_number % 2 == 0 else 1

        # Keep initial labels close to their corresponding marker.
        initial_x = row["Parameters"] * (
            1 + direction * 0.025
        )

        # Apply a small vertical displacement.
        initial_y = (
            row["RMSE"]
            + 0.0025
            + (row_number % 3) * 0.0008
        )

        label = ax.text(
            initial_x,
            initial_y,
            wrapped_name,
            fontsize=8.2,
            ha="right" if direction < 0 else "left",
            va="bottom",
            linespacing=0.95,
            bbox={
                "boxstyle": "round,pad=0.20",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.90,
            },
            zorder=6,
        )

        label_objects.append(label)

    # Save the objects for final adjustment.
    adjustment_jobs.append(
        {
            "ax": ax,
            "labels": label_objects,
            "point_x": point_x,
            "point_y": point_y,
            "exclusion_zones": point_exclusion_zones,
        }
    )


# -------------------------------------------------------------------------
# Add the figure title and highlight legend
# -------------------------------------------------------------------------

highlight_legend = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor="#d4a017",
        markeredgewidth=2.7,
        markersize=11,
        label="Lowest pooled RMSE",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markerfacecolor="none",
        markeredgecolor="#8c8c8c",
        markeredgewidth=2.7,
        markersize=11,
        label="Second-lowest pooled RMSE",
    ),
]

fig.legend(
    handles=highlight_legend,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.suptitle(
    "Model complexity versus predictive error",
    fontsize=15,
    y=0.98,
)

# Finalise all panel positions before adjusting labels.
fig.tight_layout(
    rect=(0.02, 0.10, 0.98, 0.93),
    w_pad=4.5,
)

# Finalise display coordinates before resolving label collisions.
fig.canvas.draw()


# -------------------------------------------------------------------------
# Adjust labels while keeping them as close as possible
# -------------------------------------------------------------------------

for job in adjustment_jobs:

    adjust_text(
        job["labels"],

        # Repel labels from model coordinates.
        x=job["point_x"],
        y=job["point_y"],

        # Repel labels from the complete marker areas.
        objects=job["exclusion_zones"],

        # Preserve the correct label-to-marker relationship.
        target_x=job["point_x"],
        target_y=job["point_y"],

        ax=job["ax"],
        ensure_inside_axes=True,
        expand_axes=False,

        # Prevent label overlap without introducing excessive whitespace.
        expand=(1.30, 1.55),

        # Strong enough to separate labels.
        force_text=(1.3, 2.2),

        # Prevent labels from covering dots.
        force_static=(1.8, 2.4),

        # Relatively strong pull keeps labels close to their own dots.
        force_pull=(0.18, 0.24),
        pull_threshold=8,

        # Use only a modest initial explosion.
        force_explode=(0.7, 1.0),
        explode_radius=50,

        # Prevent unnecessarily large label movements.
        max_move=(22, 28),

        # Allow convergence in crowded regions.
        iter_lim=2000,
    )

# Redraw to obtain the final adjusted label boundaries.
# Finalise display coordinates before resolving label collisions.
fig.canvas.draw()


# -------------------------------------------------------------------------
# Route short collision-aware connector lines
# -------------------------------------------------------------------------

for job in adjustment_jobs:

    add_short_collision_aware_leaders(
        ax=job["ax"],
        labels=job["labels"],
        target_x=job["point_x"],
        target_y=job["point_y"],
        point_clearance=15,
    )

# Finalise display coordinates before resolving label collisions.
fig.canvas.draw()


# -------------------------------------------------------------------------
# Save the outputs
# -------------------------------------------------------------------------

top_two_table = pd.DataFrame(top_two_rows)

top_two_table.to_csv(
    TABLE_ROOT / "table_top_two_models.csv",
    index=False,
)

fig.savefig(
    FIGURE_ROOT / "figure_parameters_vs_rmse.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_ROOT / "figure_parameters_vs_rmse.pdf",
    bbox_inches="tight",
)

plt.show()

display(
    top_two_table
    .drop(columns="model_key")
    .round(4)
)

## 7. Paired bootstrap uncertainty

Each model predicts the same held-out compounds, allowing paired error differences. Random-CV intervals resample compounds. Scaffold-CV intervals resample complete scaffold clusters. The reference is the 3D distance GNN. A positive RMSE difference means the comparison model is worse. Bootstrap intervals are uncertainty diagnostics, not corrections for all sources of training variability.

### Code used: Paired bootstrap

I align all model predictions by compound ID and target before resampling. Random CV resamples compounds; scaffold CV resamples whole scaffold groups. Every resample uses the same selected rows for every model, preserving the pairing when calculating RMSE differences.


In [ ]:
# Keep the same sampled compounds or scaffold groups for every model in each replicate.
# A one-to-one join prevents duplicate compound rows from changing paired comparisons.
def merged_predictions(method: str) -> pd.DataFrame:
    """Merge all models onto one audited compound-level target table."""
    merged = None
    for model in MODEL_ORDER:
        frame = pd.read_csv(
            prediction_path(method, model),
            dtype={"official_compound_group_id": str},
        )
        frame["official_compound_group_id"] = frame["official_compound_group_id"].str.zfill(16)
        frame = frame[["official_compound_group_id", "experimental_pKD", "predicted_pKD", "outer_fold"]].rename(
            columns={"predicted_pKD": model, "outer_fold": f"fold_{model}"}
        )
        if merged is None:
            merged = frame
        else:
            merged = merged.merge(frame, on=["official_compound_group_id", "experimental_pKD"], validate="one_to_one")
    return merged.sort_values("official_compound_group_id").reset_index(drop=True)


# Resample the correct grouping unit while sharing every draw across model columns.
def paired_bootstrap(method: str, reference: str = "3d_gnn", iterations: int = N_BOOTSTRAP) -> pd.DataFrame:
    """Estimate paired 95% CIs for RMSE differences against a reference model."""
    data = merged_predictions(method)
    truth = data["experimental_pKD"].to_numpy(float)
    predictions = np.column_stack([data[model].to_numpy(float) for model in MODEL_ORDER])
    reference_index = MODEL_ORDER.index(reference)
    rng = np.random.default_rng(RNG_SEED + (0 if method == "random" else 10_000))

    if method == "random":
        # Random CV treats one compound as the resampling unit.
        sampling_units = [np.array([index], dtype=int) for index in range(len(data))]
    else:
        scaffold_map = assignments["scaffold"].set_index("official_compound_group_id")["scaffold"]
        scaffolds = np.asarray([scaffold_map.loc[compound_id] for compound_id in data["official_compound_group_id"]])
        # Scaffold CV keeps each scaffold group's compounds together within a draw.
        sampling_units = [np.flatnonzero(scaffolds == scaffold) for scaffold in pd.unique(scaffolds)]

    bootstrap_differences = np.empty((iterations, len(MODEL_ORDER)), dtype=float)
    for iteration in range(iterations):
        chosen = rng.integers(0, len(sampling_units), size=len(sampling_units))
        # Reuse this resampled row index for all model predictions.
        indices = np.concatenate([sampling_units[position] for position in chosen])
        sampled_truth = truth[indices]
        sampled_predictions = predictions[indices]
        rmses = np.sqrt(np.mean((sampled_predictions - sampled_truth[:, None]) ** 2, axis=0))
        # Subtract the reference within the replicate to retain pairing.
        bootstrap_differences[iteration] = rmses - rmses[reference_index]

    observed_rmses = np.sqrt(np.mean((predictions - truth[:, None]) ** 2, axis=0))
    observed_delta = observed_rmses - observed_rmses[reference_index]
    rows = []
    for model_index, model in enumerate(MODEL_ORDER):
        # Use percentile bounds from the distribution of paired RMSE differences.
        lower, upper = np.quantile(bootstrap_differences[:, model_index], [0.025, 0.975])
        rows.append({
            "CV method": method,
            "Reference": MODEL_LABELS[reference],
            "Model": MODEL_LABELS[model],
            "RMSE difference": observed_delta[model_index],
            "95% CI lower": lower,
            "95% CI upper": upper,
            "Interval excludes zero": bool(lower > 0 or upper < 0),
        })
    return pd.DataFrame(rows)


bootstrap_table = pd.concat([
    paired_bootstrap("random"),
    paired_bootstrap("scaffold"),
], ignore_index=True)
bootstrap_table.to_csv(TABLE_ROOT / "table_paired_bootstrap_vs_3d_gnn.csv", index=False)
display(bootstrap_table.round(4))

### Code used: Bootstrap interval plot

I display the saved paired differences and percentile intervals against the 3D distance GNN reference. The plot omits the reference's self-comparison and uses shared horizontal limits.


In [ ]:
# Draw uncertainty for paired differences, not intervals around independent model scores.
# -------------------------------------------------------------------------
# Prepare the paired-bootstrap results
# -------------------------------------------------------------------------

# Obtain the displayed name of the reference model.
reference_label = bootstrap_table["Reference"].iloc[0]

# Remove the uninformative reference-versus-itself rows.
forest_data = bootstrap_table.loc[
    bootstrap_table["Model"] != reference_label
].copy()

# Preserve the original model order.
comparison_model_order = [
    MODEL_LABELS[model]
    for model in MODEL_ORDER
    if MODEL_LABELS[model] != reference_label
]

# Determine common x-axis limits across random and scaffold CV.
minimum_ci = min(
    0,
    forest_data["95% CI lower"].min(),
)

maximum_ci = max(
    0,
    forest_data["95% CI upper"].max(),
)

x_range = maximum_ci - minimum_ci
x_padding = max(
    0.008,
    x_range * 0.10,
)

common_x_limits = (
    minimum_ci - x_padding,
    maximum_ci + x_padding,
)


# -------------------------------------------------------------------------
# Create the forest plot
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.5, 6.2),
    sharex=True,
    sharey=False,
)

# Produce separate random-CV and scaffold-CV panels.
for ax, method in zip(
    axes,
    ("random", "scaffold"),
):

    # Select results for the current CV method.
    method_data = forest_data.loc[
        forest_data["CV method"] == method
    ].copy()

    # Apply the original model order.
    method_data = (
        method_data
        .set_index("Model")
        .loc[comparison_model_order]
        .reset_index()
    )

    # Define one vertical position for each comparison model.
    y_positions = np.arange(
        len(method_data)
    )

    # ---------------------------------------------------------------------
    # Plot the confidence interval for each model
    # ---------------------------------------------------------------------

    for y_position, (_, row) in enumerate(
        method_data.iterrows()
    ):

        # Read the observed RMSE difference and bootstrap limits.
        difference = float(
            row["RMSE difference"]
        )

        lower = float(
            row["95% CI lower"]
        )

        upper = float(
            row["95% CI upper"]
        )

        excludes_zero = bool(
            row["Interval excludes zero"]
        )

        # Convert confidence limits into asymmetric error lengths.
        lower_error = difference - lower
        upper_error = upper - difference

        # Filled blue points indicate an interval excluding zero.
        if excludes_zero:
            marker_face = "#2f6f9f"
            marker_edge = "#1f4e70"
            line_colour = "#2f6f9f"

        # Hollow grey points indicate an interval containing zero.
        else:
            marker_face = "white"
            marker_edge = "#666666"
            line_colour = "#888888"

        # Plot the RMSE difference and its 95% confidence interval.
        ax.errorbar(
            difference,
            y_position,
            xerr=np.array(
                [
                    [lower_error],
                    [upper_error],
                ]
            ),
            fmt="o",
            markersize=7.5,
            markerfacecolor=marker_face,
            markeredgecolor=marker_edge,
            markeredgewidth=1.3,
            ecolor=line_colour,
            elinewidth=1.7,
            capsize=4,
            capthick=1.4,
            zorder=3,
        )

        # Place positive values to the right and negative values to the left.
        if difference >= 0:
            text_offset = 7
            text_alignment = "left"
        else:
            text_offset = -7
            text_alignment = "right"

        # Add the observed numerical difference.
        ax.annotate(
            f"{difference:+.3f}",
            xy=(
                difference,
                y_position,
            ),
            xytext=(
                text_offset,
                7,
            ),
            textcoords="offset points",
            ha=text_alignment,
            va="bottom",
            fontsize=8,
        )

    # ---------------------------------------------------------------------
    # Format the current panel
    # ---------------------------------------------------------------------

    # A difference of zero means equal pooled RMSE.
    ax.axvline(
        0,
        color="black",
        linewidth=1.2,
        linestyle="--",
        zorder=1,
    )

    # Add the model names to the y-axis.
    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        method_data["Model"]
    )

    # Place the first comparison model at the top.
    ax.invert_yaxis()

    # Use identical horizontal limits in both panels.
    ax.set_xlim(common_x_limits)

    # Label both axes.
    ax.set_xlabel(
        "RMSE difference relative to 3D distance GNN (pKD)",
        labelpad=9,
    )

    ax.set_ylabel(
        "Comparison model",
        labelpad=9,
    )

    # Add the panel title.
    ax.set_title(
        f"{method.capitalize()} five-fold CV",
        pad=22,
    )

    # Explain the direction of the difference.
    ax.text(
        0.01,
        1.02,
        "← Comparison model better",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.5,
        color="#444444",
    )

    ax.text(
        0.99,
        1.02,
        "3D distance GNN better →",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8.5,
        color="#444444",
    )

    # Add horizontal guides only.
    ax.grid(
        axis="y",
        alpha=0.18,
        linewidth=0.7,
        zorder=0,
    )

    # Remove unnecessary borders.
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# -------------------------------------------------------------------------
# Add the title and legend
# -------------------------------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="-",
        color="#2f6f9f",
        markerfacecolor="#2f6f9f",
        markeredgecolor="#1f4e70",
        markersize=7.5,
        label="95% interval excludes zero",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="-",
        color="#888888",
        markerfacecolor="white",
        markeredgecolor="#666666",
        markersize=7.5,
        label="95% interval includes zero",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.suptitle(
    "Paired RMSE differences relative to the 3D distance GNN",
    fontsize=14,
    y=0.98,
)

# Reserve space for the title, direction labels and bottom legend.
fig.tight_layout(
    rect=(0.02, 0.10, 0.98, 0.91),
    w_pad=4.0,
)


# -------------------------------------------------------------------------
# Save and display the figure
# -------------------------------------------------------------------------

fig.savefig(
    FIGURE_ROOT / "figure_paired_bootstrap_forest.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_ROOT / "figure_paired_bootstrap_forest.pdf",
    bbox_inches="tight",
)

plt.show()

## 8. Training and validation behaviour

The light lines show individual outer folds. Training and validation losses are normalised Huber losses and are therefore suitable for within-protocol learning diagnostics, not direct comparison with RMSE in pKD units.

### Code used: Selected-model learning curves

I read each fold's training history, locate its minimum validation loss and plot the selected model families. Best-epoch labels are moved for readability; the checkpoint criterion is not changed.


In [ ]:
# Select the displayed epoch from validation loss, never from held-out test error.
# -------------------------------------------------------------------------
# Resolve result directories
# -------------------------------------------------------------------------

# Account for the different masked/unmasked output folder conventions.
def result_directory(
    method: str,
    fold: int,
    model: str,
) -> Path:
    """Return the directory containing one completed training run."""

    root = RESULT_ROOTS[method] / f"fold_{fold}"

    if model in {
        "morgan_mlp",
        "2d_gnn",
        "3d_gnn",
        "3d_alignn",
    }:
        return (
            root
            / "unmasked"
            / model
            / method
            / "seed_123"
        )

    if model == "adapted_mgt":
        return (
            root
            / "unmasked"
            / "adapted_mgt"
            / method
            / "seed_123"
        )

    if model == "masked_alignn":
        return (
            root
            / "masked_alignn"
            / "3d_alignn"
            / method
            / "seed_123"
        )

    if model == "masked_mgt":
        return (
            root
            / "masked_mgt"
            / method
            / "seed_123"
        )

    raise KeyError(
        f"Unknown model key: {model}"
    )


# -------------------------------------------------------------------------
# Load the five fold histories
# -------------------------------------------------------------------------

# Read each outer fold independently so epoch and checkpoint annotations remain traceable.
def load_histories(
    method: str,
    model: str,
) -> list[pd.DataFrame]:
    """Load all five histories for one model and CV method."""

    histories = []

    for fold in range(5):

        history_path = (
            result_directory(
                method,
                fold,
                model,
            )
            / "history.csv"
        )

        history = pd.read_csv(history_path)

        history = (
            history
            .sort_values("epoch")
            .reset_index(drop=True)
        )

        history["outer_fold"] = fold
        histories.append(history)

    return histories


# -------------------------------------------------------------------------
# Plot configuration
# -------------------------------------------------------------------------

# Restrict the main-text panels; the next cell covers the full model set.
selected_models = [
    "3d_gnn",
    "3d_alignn",
    "adapted_mgt",
    "masked_mgt",
]

# Bright orange-yellow dotted training curves.
training_colour = "#FF9900"

# Muted blue validation curves.
validation_colour = "#7FA6C2"

# Distinct colours for the selected epochs.
best_epoch_colours = {
    0: "#D55E00",
    1: "#009E73",
    2: "#7B61A8",
    3: "#C44E52",
    4: "#4D4D4D",
}

fold_markers = {
    0: "o",
    1: "s",
    2: "^",
    3: "D",
    4: "P",
}

# Different initial line lengths reduce label crowding.
line_extension_fractions = {
    0: 0.10,
    1: 0.14,
    2: 0.18,
    3: 0.22,
    4: 0.26,
}

best_epoch_rows = []


# -------------------------------------------------------------------------
# Produce separate random and scaffold 2×2 figures
# -------------------------------------------------------------------------

for method in (
    "random",
    "scaffold",
):

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(20, 14.5),
        sharex=False,
        sharey=False,
    )

    axes = axes.flatten()

    # Store label-adjustment information for this figure.
    adjustment_jobs = []

    for ax, model in zip(
        axes,
        selected_models,
    ):

        # Store selected epochs until the y-axis range is available.
        selected_epochs = []

        # -------------------------------------------------------------
        # Plot all five fold histories
        # -------------------------------------------------------------

        for history in load_histories(
            method,
            model,
        ):

            fold = int(
                history["outer_fold"].iloc[0]
            )

            valid_history = history.dropna(
                subset=[
                    "epoch",
                    "train_loss",
                    "validation_loss",
                ]
            ).copy()

            # Dotted training curves.
            ax.plot(
                valid_history["epoch"],
                valid_history["train_loss"],
                color=training_colour,
                linewidth=2.1,
                linestyle=":",
                alpha=0.80,
                zorder=1,
            )

            # Solid validation curves.
            ax.plot(
                valid_history["epoch"],
                valid_history["validation_loss"],
                color=validation_colour,
                linewidth=2.2,
                linestyle="-",
                alpha=0.78,
                zorder=2,
            )

            # Find the minimum validation loss.
            best_index = (
                valid_history["validation_loss"]
                .idxmin()
            )

            best_epoch = float(
                valid_history.loc[
                    best_index,
                    "epoch",
                ]
            )

            best_validation_loss = float(
                valid_history.loc[
                    best_index,
                    "validation_loss",
                ]
            )

            training_loss_at_best_epoch = float(
                valid_history.loc[
                    best_index,
                    "train_loss",
                ]
            )

            rounded_epoch = int(
                round(best_epoch)
            )

            selected_epochs.append(
                {
                    "fold": fold,
                    "epoch": best_epoch,
                    "rounded_epoch": rounded_epoch,
                    "validation_loss": best_validation_loss,
                }
            )

            best_epoch_rows.append(
                {
                    "CV method": method,
                    "Model": MODEL_LABELS[model],
                    "Fold": fold,
                    "Best epoch": rounded_epoch,
                    "Minimum validation loss": (
                        best_validation_loss
                    ),
                    "Training loss at best epoch": (
                        training_loss_at_best_epoch
                    ),
                }
            )

        # -------------------------------------------------------------
        # Finalise the panel range
        # -------------------------------------------------------------

        ax.relim()
        ax.autoscale_view()

        ax.margins(
            x=0.06,
            y=0.10,
        )

        y_minimum, y_maximum = ax.get_ylim()
        y_range = y_maximum - y_minimum

        # Store text objects and their original line endpoints.
        label_objects = []
        label_target_x = []
        label_target_y = []

        # -------------------------------------------------------------
        # Add selected-epoch lines, markers and labels
        # -------------------------------------------------------------

        for selected in selected_epochs:

            fold = selected["fold"]
            best_epoch = selected["epoch"]
            best_loss = selected["validation_loss"]
            rounded_epoch = selected["rounded_epoch"]

            # Calculate a short vertical line.
            line_top = (
                best_loss
                + line_extension_fractions[fold] * y_range
            )

            maximum_line_top = (
                y_maximum - 0.15 * y_range
            )

            line_top = min(
                line_top,
                maximum_line_top,
            )

            line_top = max(
                line_top,
                best_loss + 0.07 * y_range,
            )

            # Draw the short vertical epoch indicator.
            ax.vlines(
                best_epoch,
                ymin=best_loss,
                ymax=line_top,
                color=best_epoch_colours[fold],
                linewidth=1.7,
                linestyle="-",
                alpha=0.90,
                zorder=3,
            )

            # Mark the best validation-loss point.
            ax.scatter(
                best_epoch,
                best_loss,
                s=115,
                marker=fold_markers[fold],
                facecolor=best_epoch_colours[fold],
                edgecolor="white",
                linewidth=1.5,
                zorder=5,
            )

            # Create the initial label at the end of the vertical line.
            label = ax.text(
                best_epoch,
                line_top,
                f"Fold: {fold}, Epoch: {rounded_epoch}",
                rotation=90,
                rotation_mode="anchor",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="semibold",
                color=best_epoch_colours[fold],
                bbox={
                    "boxstyle": "round,pad=0.20",
                    "facecolor": "white",
                    "edgecolor": best_epoch_colours[fold],
                    "linewidth": 0.8,
                    "alpha": 0.95,
                },
                clip_on=True,
                zorder=6,
            )

            label_objects.append(label)
            label_target_x.append(best_epoch)
            label_target_y.append(line_top)

        # Add invisible exclusion areas around line endpoints.
        endpoint_exclusion_zones = ax.scatter(
            label_target_x,
            label_target_y,
            s=180,
            facecolors="none",
            edgecolors="none",
            alpha=0,
            zorder=2,
        )

        adjustment_jobs.append(
            {
                "ax": ax,
                "labels": label_objects,
                "target_x": label_target_x,
                "target_y": label_target_y,
                "exclusion_zones": endpoint_exclusion_zones,
            }
        )

        # -------------------------------------------------------------
        # Format the panel
        # -------------------------------------------------------------

        ax.set_title(
            MODEL_LABELS[model],
            fontsize=18,
            pad=16,
        )

        ax.set_xlabel(
            "Epoch",
            fontsize=14,
            labelpad=10,
        )

        ax.set_ylabel(
            "Normalised Huber loss",
            fontsize=14,
            labelpad=11,
        )

        ax.tick_params(
            axis="both",
            labelsize=12,
        )

        ax.grid(
            alpha=0.15,
            linewidth=0.8,
            zorder=0,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)


    # ---------------------------------------------------------------------
    # Create the legend
    # ---------------------------------------------------------------------

    legend_handles = [
        Line2D(
            [0],
            [0],
            color=training_colour,
            linewidth=2.4,
            linestyle=":",
            label="Training loss",
        ),
        Line2D(
            [0],
            [0],
            color=validation_colour,
            linewidth=2.4,
            linestyle="-",
            label="Validation loss",
        ),
    ]

    for fold in range(5):
        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=best_epoch_colours[fold],
                linewidth=1.8,
                linestyle="-",
                marker=fold_markers[fold],
                markerfacecolor=best_epoch_colours[fold],
                markeredgecolor="white",
                markersize=9,
                label=f"Fold {fold} selected epoch",
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=11,
        bbox_to_anchor=(0.5, 0.025),
    )

    fig.suptitle(
        (
            f"Training and validation curves: "
            f"{method.capitalize()} five-fold cross-validation"
        ),
        fontsize=22,
        y=0.985,
    )

    # Finalise the axes positions before adjusting text.
    fig.subplots_adjust(
        left=0.075,
        right=0.98,
        bottom=0.13,
        top=0.91,
        wspace=0.20,
        hspace=0.34,
    )

    fig.canvas.draw()


    # ---------------------------------------------------------------------
    # Adjust overlapping labels and boxes
    # ---------------------------------------------------------------------

    for job in adjustment_jobs:

        if len(job["labels"]) < 2:
            continue

        adjust_text(
            job["labels"],

            # Treat all label endpoints as fixed objects to avoid.
            x=job["target_x"],
            y=job["target_y"],

            # Avoid the complete exclusion region around each endpoint.
            objects=job["exclusion_zones"],

            # Preserve each label's connection to its epoch line.
            target_x=job["target_x"],
            target_y=job["target_y"],

            ax=job["ax"],

            # Keep every text box inside the plotting panel.
            ensure_inside_axes=True,
            expand_axes=False,

            # Increase the separation between rendered label boxes.
            expand=(1.30, 1.45),

            # Strongly repel labels from one another.
            force_text=(1.8, 2.2),

            # Repel labels from epoch-line endpoints.
            force_static=(1.5, 1.8),

            # Keep adjusted labels reasonably close to their own lines.
            force_pull=(0.10, 0.13),
            pull_threshold=8,

            # Separate labels that begin at similar epochs.
            force_explode=(1.1, 1.4),
            explode_radius=55,

            # Permit enough movement to resolve crowded labels.
            max_move=(24, 30),
            iter_lim=2000,

            # Reduce crossing connector lines.
            prevent_crossings=True,

            # Add a connector only after a meaningful movement.
            min_arrow_len=7,

            arrowprops={
                "arrowstyle": "-",
                "color": "#707070",
                "linewidth": 0.55,
                "alpha": 0.75,
            },
        )

    fig.canvas.draw()


    # ---------------------------------------------------------------------
    # Save each figure separately
    # ---------------------------------------------------------------------

    fig.savefig(
        FIGURE_ROOT
        / f"figure_learning_curves_{method}_2x2.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        FIGURE_ROOT
        / f"figure_learning_curves_{method}_2x2.pdf",
        bbox_inches="tight",
    )

    plt.show()


# -------------------------------------------------------------------------
# Save the selected-epoch table
# -------------------------------------------------------------------------

best_epoch_table = pd.DataFrame(
    best_epoch_rows
)

best_epoch_table.to_csv(
    TABLE_ROOT
    / "table_selected_model_best_epochs.csv",
    index=False,
)

display(
    best_epoch_table.round(4)
)

### Code used: All-model learning-curve appendix

I reuse the history loader to draw learning curves for all seven configurations under both CV designs. A separate table records the displayed best-validation epochs.


In [ ]:
# Extend the same learning-curve inspection to the complete model set.
# -------------------------------------------------------------------------
# Appendix figures: every model, separated by CV design
# -------------------------------------------------------------------------

appendix_best_epoch_rows = []

for method in (
    "random",
    "scaffold",
):

    # Calculate the number of rows needed for two columns.
    number_of_columns = 2
    number_of_rows = int(
        np.ceil(
            len(MODEL_ORDER)
            / number_of_columns
        )
    )

    # Create a larger appendix figure for readable panels.
    fig, axes = plt.subplots(
        number_of_rows,
        number_of_columns,
        figsize=(18, 22),
        sharex=False,
        sharey=False,
    )

    axes = np.asarray(axes).ravel()

    # Store label-adjustment information for every panel.
    adjustment_jobs = []

    # ---------------------------------------------------------------------
    # Plot every model
    # ---------------------------------------------------------------------

    for ax, model in zip(
        axes,
        MODEL_ORDER,
    ):

        selected_epochs = []

        # Plot all five outer-fold histories.
        for history in load_histories(
            method,
            model,
        ):

            fold = int(
                history["outer_fold"].iloc[0]
            )

            valid_history = history.dropna(
                subset=[
                    "epoch",
                    "train_loss",
                    "validation_loss",
                ]
            ).copy()

            # Plot training loss as a dotted orange-yellow line.
            ax.plot(
                valid_history["epoch"],
                valid_history["train_loss"],
                color=training_colour,
                alpha=0.72,
                linewidth=1.5,
                linestyle=":",
                zorder=1,
            )

            # Plot validation loss as a solid muted-blue line.
            ax.plot(
                valid_history["epoch"],
                valid_history["validation_loss"],
                color=validation_colour,
                alpha=0.76,
                linewidth=1.6,
                linestyle="-",
                zorder=2,
            )

            # Identify the epoch with the lowest validation loss.
            best_index = (
                valid_history["validation_loss"]
                .idxmin()
            )

            best_epoch = float(
                valid_history.loc[
                    best_index,
                    "epoch",
                ]
            )

            best_validation_loss = float(
                valid_history.loc[
                    best_index,
                    "validation_loss",
                ]
            )

            training_loss_at_best_epoch = float(
                valid_history.loc[
                    best_index,
                    "train_loss",
                ]
            )

            rounded_epoch = int(
                round(best_epoch)
            )

            selected_epochs.append(
                {
                    "fold": fold,
                    "epoch": best_epoch,
                    "rounded_epoch": rounded_epoch,
                    "validation_loss": best_validation_loss,
                }
            )

            appendix_best_epoch_rows.append(
                {
                    "CV method": method,
                    "Model": MODEL_LABELS[model],
                    "Fold": fold,
                    "Best epoch": rounded_epoch,
                    "Minimum validation loss": (
                        best_validation_loss
                    ),
                    "Training loss at best epoch": (
                        training_loss_at_best_epoch
                    ),
                }
            )

        # -------------------------------------------------------------
        # Calculate the panel limits
        # -------------------------------------------------------------

        ax.relim()
        ax.autoscale_view()

        ax.margins(
            x=0.06,
            y=0.10,
        )

        y_minimum, y_maximum = ax.get_ylim()
        y_range = y_maximum - y_minimum

        label_objects = []
        label_target_x = []
        label_target_y = []

        # -------------------------------------------------------------
        # Add short selected-epoch indicators
        # -------------------------------------------------------------

        for selected in selected_epochs:

            fold = selected["fold"]
            best_epoch = selected["epoch"]
            best_loss = selected["validation_loss"]
            rounded_epoch = selected["rounded_epoch"]

            # Calculate a short fold-specific line length.
            line_top = (
                best_loss
                + line_extension_fractions[fold] * y_range
            )

            maximum_line_top = (
                y_maximum - 0.15 * y_range
            )

            line_top = min(
                line_top,
                maximum_line_top,
            )

            line_top = max(
                line_top,
                best_loss + 0.07 * y_range,
            )

            # Draw the short vertical segment.
            ax.vlines(
                best_epoch,
                ymin=best_loss,
                ymax=line_top,
                color=best_epoch_colours[fold],
                linewidth=1.4,
                linestyle="-",
                alpha=0.88,
                zorder=3,
            )

            # Mark the minimum validation-loss point.
            ax.scatter(
                best_epoch,
                best_loss,
                s=85,
                marker=fold_markers[fold],
                facecolor=best_epoch_colours[fold],
                edgecolor="white",
                linewidth=1.1,
                zorder=5,
            )

            # Add an initial single-line rotated label.
            label = ax.text(
                best_epoch,
                line_top,
                f"Fold: {fold}, Epoch: {rounded_epoch}",
                rotation=90,
                rotation_mode="anchor",
                ha="center",
                va="bottom",
                fontsize=7.5,
                fontweight="semibold",
                color=best_epoch_colours[fold],
                bbox={
                    "boxstyle": "round,pad=0.16",
                    "facecolor": "white",
                    "edgecolor": best_epoch_colours[fold],
                    "linewidth": 0.65,
                    "alpha": 0.94,
                },
                clip_on=True,
                zorder=6,
            )

            label_objects.append(label)
            label_target_x.append(best_epoch)
            label_target_y.append(line_top)

        # Create invisible exclusion regions around the line endpoints.
        endpoint_exclusion_zones = ax.scatter(
            label_target_x,
            label_target_y,
            s=150,
            facecolors="none",
            edgecolors="none",
            alpha=0,
            zorder=2,
        )

        adjustment_jobs.append(
            {
                "ax": ax,
                "labels": label_objects,
                "target_x": label_target_x,
                "target_y": label_target_y,
                "exclusion_zones": endpoint_exclusion_zones,
            }
        )

        # -------------------------------------------------------------
        # Format the panel
        # -------------------------------------------------------------

        ax.set_title(
            MODEL_LABELS[model],
            fontsize=14,
            pad=12,
        )

        ax.set_xlabel(
            "Epoch",
            fontsize=11,
        )

        ax.set_ylabel(
            "Normalised Huber loss",
            fontsize=11,
        )

        ax.tick_params(
            axis="both",
            labelsize=9.5,
        )

        ax.grid(
            alpha=0.15,
            linewidth=0.7,
            zorder=0,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)


    # ---------------------------------------------------------------------
    # Hide any unused subplot
    # ---------------------------------------------------------------------

    for unused_axis in axes[
        len(MODEL_ORDER):
    ]:
        unused_axis.axis("off")


    # ---------------------------------------------------------------------
    # Add the figure title and legend
    # ---------------------------------------------------------------------

    legend_handles = [
        Line2D(
            [0],
            [0],
            color=training_colour,
            linewidth=2.0,
            linestyle=":",
            label="Training loss",
        ),
        Line2D(
            [0],
            [0],
            color=validation_colour,
            linewidth=2.0,
            linestyle="-",
            label="Validation loss",
        ),
    ]

    for fold in range(5):
        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=best_epoch_colours[fold],
                linewidth=1.5,
                marker=fold_markers[fold],
                markerfacecolor=best_epoch_colours[fold],
                markeredgecolor="white",
                markersize=8,
                label=f"Fold {fold} selected epoch",
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=4,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, 0.018),
    )

    fig.suptitle(
        (
            f"All models: {method.capitalize()} "
            "five-fold cross-validation learning curves"
        ),
        fontsize=18,
        y=0.985,
    )

    # Finalise all subplot positions before adjusting labels.
    fig.tight_layout(
        rect=(
            0.02,
            0.07,
            0.98,
            0.96,
        ),
        h_pad=2.5,
        w_pad=2.2,
    )

    fig.canvas.draw()


    # ---------------------------------------------------------------------
    # Prevent overlap between epoch-label boxes
    # ---------------------------------------------------------------------

    for job in adjustment_jobs:

        if len(job["labels"]) < 2:
            continue

        adjust_text(
            job["labels"],
            x=job["target_x"],
            y=job["target_y"],
            objects=job["exclusion_zones"],
            target_x=job["target_x"],
            target_y=job["target_y"],
            ax=job["ax"],

            # Keep labels within the panel.
            ensure_inside_axes=True,
            expand_axes=False,

            # Separate complete rendered label boxes.
            expand=(1.25, 1.40),
            force_text=(1.6, 2.0),

            # Prevent labels from covering line endpoints.
            force_static=(1.4, 1.7),

            # Keep labels close to their corresponding epochs.
            force_pull=(0.10, 0.13),
            pull_threshold=7,

            # Separate initially crowded labels.
            force_explode=(1.0, 1.3),
            explode_radius=45,

            max_move=(20, 26),
            iter_lim=1500,
            prevent_crossings=True,
            min_arrow_len=6,

            # Draw a short connector after displacement.
            arrowprops={
                "arrowstyle": "-",
                "color": "#707070",
                "linewidth": 0.5,
                "alpha": 0.70,
            },
        )

    fig.canvas.draw()


    # ---------------------------------------------------------------------
    # Save the appendix figure
    # ---------------------------------------------------------------------

    fig.savefig(
        FIGURE_ROOT
        / f"appendix_learning_curves_{method}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        FIGURE_ROOT
        / f"appendix_learning_curves_{method}.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)


# -------------------------------------------------------------------------
# Save the appendix best-epoch table
# -------------------------------------------------------------------------

appendix_best_epoch_table = pd.DataFrame(
    appendix_best_epoch_rows
)

appendix_best_epoch_table.to_csv(
    TABLE_ROOT
    / "table_appendix_all_model_best_epochs.csv",
    index=False,
)

print(
    "Saved complete collision-aware learning-curve appendix figures."
)

## 9. Predicted versus experimental affinity

The best pooled-RMSE models are shown: 3D ALIGNN for random CV and the 3D distance GNN for scaffold CV. Every point is a compound's single out-of-fold prediction.

### Code used: Selected-model prediction plots

I compare pooled compound predictions with experimental pKD for the model keys specified in this cell. The identity line shows exact agreement; this is not a new model-selection or training step.


In [ ]:
# These selected model keys are explicit settings for the reporting figure.
# The reporting choices are fixed here rather than selected automatically on rerun.
best_models = {"random": "3d_alignn", "scaffold": "3d_gnn"}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharex=True, sharey=True)
for ax, method in zip(axes, ("random", "scaffold")):
    model = best_models[method]
    data = pd.read_csv(prediction_path(method, model))
    observed = data["experimental_pKD"].to_numpy(float)
    predicted = data["predicted_pKD"].to_numpy(float)
    limits = [min(observed.min(), predicted.min()) - 0.15, max(observed.max(), predicted.max()) + 0.15]
    ax.scatter(observed, predicted, s=28, alpha=0.65, color=MODEL_COLOURS[model], edgecolor="white", linewidth=0.3)
    ax.plot(limits, limits, color="black", linestyle="--", linewidth=1.2, label="Ideal")
    slope, intercept = np.polyfit(observed, predicted, 1)
    x_line = np.linspace(*limits, 100)
    ax.plot(x_line, slope * x_line + intercept, color="#e15759", linewidth=1.5, label="Linear fit")
    metric = cv_summaries[method]["pooled_out_of_fold_metrics"][model]
    annotation = f"n = {len(data)}\nRMSE = {metric['rmse']:.3f}\nR² = {metric['r2']:.3f}\nSpearman = {metric['spearman_r']:.3f}"
    ax.text(0.04, 0.96, annotation, transform=ax.transAxes, va="top", bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85, "edgecolor": "#cccccc"})
    ax.set(xlim=limits, ylim=limits, xlabel="Experimental pKD", ylabel="Predicted pKD", title=f"{method.capitalize()}: {MODEL_LABELS[model]}")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(frameon=False, loc="lower right")
fig.suptitle("Pooled out-of-fold affinity predictions", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURE_ROOT / "figure_predicted_vs_experimental_best_models.png", bbox_inches="tight")
fig.savefig(FIGURE_ROOT / "figure_predicted_vs_experimental_best_models.pdf", bbox_inches="tight")
plt.show()

### Code used: All-model prediction appendix

I apply the same prediction-plot format across all models so the random and scaffold panels are directly comparable. The highlighted model keys are specified in the code.


In [ ]:
# Reuse the prediction axes and layout across all seven configurations.
# -------------------------------------------------------------------------
# Define the model groups
# -------------------------------------------------------------------------

baseline_models = [
    "morgan_mlp",
    "2d_gnn",
    "3d_gnn",
    "3d_alignn",
]

mgt_and_masked_models = [
    "adapted_mgt",
    "masked_alignn",
    "masked_mgt",
]

plot_model_order = (
    baseline_models
    + mgt_and_masked_models
)


# -------------------------------------------------------------------------
# Identify the lowest-RMSE model under each CV design
# -------------------------------------------------------------------------

best_models = {
    "random": "3d_alignn",
    "scaffold": "3d_gnn",
}

best_model_colour = "#D4A017"


# -------------------------------------------------------------------------
# Create separate random and scaffold figures
# -------------------------------------------------------------------------

for method in (
    "random",
    "scaffold",
):

    # ---------------------------------------------------------------------
    # Load and cache prediction data
    # ---------------------------------------------------------------------

    prediction_data = {}
    all_values = []

    for model in plot_model_order:

        data = pd.read_csv(
            prediction_path(
                method,
                model,
            )
        )

        observed = data[
            "experimental_pKD"
        ].to_numpy(float)

        predicted = data[
            "predicted_pKD"
        ].to_numpy(float)

        prediction_data[model] = {
            "data": data,
            "observed": observed,
            "predicted": predicted,
        }

        all_values.extend(observed)
        all_values.extend(predicted)

    # Use identical limits in every panel.
    all_values = np.asarray(
        all_values,
        dtype=float,
    )

    common_limits = [
        all_values.min() - 0.15,
        all_values.max() + 0.15,
    ]


    # ---------------------------------------------------------------------
    # Create the figure and main layout
    # ---------------------------------------------------------------------

    fig = plt.figure(
        figsize=(22, 14.5)
    )

    # Four layout rows:
    # 1. Baseline heading
    # 2. Four baseline model panels
    # 3. MGT/masking heading
    # 4. Three MGT/masking panels
    outer_grid = fig.add_gridspec(
        4,
        1,
        height_ratios=[
            0.075,
            1.0,
            0.075,
            1.0,
        ],
        left=0.045,
        right=0.985,
        bottom=0.085,
        top=0.925,
        hspace=0.16,
    )


    # ---------------------------------------------------------------------
    # Add the baseline-row heading
    # ---------------------------------------------------------------------

    baseline_heading_ax = fig.add_subplot(
        outer_grid[0]
    )

    baseline_heading_ax.axis("off")

    baseline_heading_ax.text(
        0.5,
        0.50,
        "Baseline and controlled graph models",
        ha="center",
        va="center",
        fontsize=16,
        fontweight="semibold",
    )


    # ---------------------------------------------------------------------
    # Create four baseline panels
    # ---------------------------------------------------------------------

    baseline_grid = outer_grid[1].subgridspec(
        1,
        4,
        wspace=0.32,
    )

    baseline_axes = [
        fig.add_subplot(
            baseline_grid[0, index]
        )
        for index in range(4)
    ]


    # ---------------------------------------------------------------------
    # Add the MGT/masking-row heading
    # ---------------------------------------------------------------------

    mgt_heading_ax = fig.add_subplot(
        outer_grid[2]
    )

    mgt_heading_ax.axis("off")

    mgt_heading_ax.text(
        0.5,
        0.50,
        "Adapted MGT and masked-pretraining models",
        ha="center",
        va="center",
        fontsize=16,
        fontweight="semibold",
    )


    # ---------------------------------------------------------------------
    # Create three MGT/masking panels
    # ---------------------------------------------------------------------

    mgt_grid = outer_grid[3].subgridspec(
        1,
        3,
        wspace=0.28,
    )

    mgt_axes = [
        fig.add_subplot(
            mgt_grid[0, index]
        )
        for index in range(3)
    ]


    # Map each model to its corresponding axes.
    model_axes = {
        model: ax
        for model, ax in zip(
            plot_model_order,
            baseline_axes + mgt_axes,
        )
    }


    # ---------------------------------------------------------------------
    # Plot every model
    # ---------------------------------------------------------------------

    for model in plot_model_order:

        ax = model_axes[model]

        data = prediction_data[model]["data"]
        observed = prediction_data[model]["observed"]
        predicted = prediction_data[model]["predicted"]

        # -------------------------------------------------------------
        # Plot the predictions
        # -------------------------------------------------------------

        ax.scatter(
            observed,
            predicted,
            s=33,
            alpha=0.62,
            color=MODEL_COLOURS[model],
            edgecolor="white",
            linewidth=0.35,
            zorder=2,
        )

        # Plot the ideal y=x line.
        ax.plot(
            common_limits,
            common_limits,
            color="black",
            linestyle="--",
            linewidth=1.3,
            zorder=1,
        )

        # Fit a linear relationship.
        slope, intercept = np.polyfit(
            observed,
            predicted,
            1,
        )

        x_line = np.linspace(
            common_limits[0],
            common_limits[1],
            200,
        )

        # Plot the fitted relationship.
        ax.plot(
            x_line,
            slope * x_line + intercept,
            color="#e15759",
            linewidth=1.8,
            zorder=3,
        )


        # -------------------------------------------------------------
        # Add the performance annotation
        # -------------------------------------------------------------

        metric = cv_summaries[
            method
        ][
            "pooled_out_of_fold_metrics"
        ][
            model
        ]

        annotation = (
            f"n = {len(data)}\n"
            f"RMSE = {metric['rmse']:.3f}\n"
            f"R² = {metric['r2']:.3f}\n"
            f"Spearman = {metric['spearman_r']:.3f}"
        )

        ax.text(
            0.035,
            0.965,
            annotation,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=11,
            linespacing=1.15,
            bbox={
                "boxstyle": "round,pad=0.42",
                "facecolor": "white",
                "edgecolor": "#bdbdbd",
                "linewidth": 0.9,
                "alpha": 0.92,
            },
            zorder=5,
        )


        # -------------------------------------------------------------
        # Format the axes
        # -------------------------------------------------------------

        ax.set_xlim(common_limits)
        ax.set_ylim(common_limits)

        ax.set_xlabel(
            "Experimental pKD",
            fontsize=12,
            labelpad=7,
        )

        ax.set_ylabel(
            "Predicted pKD",
            fontsize=12,
            labelpad=7,
        )

        ax.set_title(
            MODEL_LABELS[model],
            fontsize=15,
            pad=13,
        )

        ax.set_aspect(
            "equal",
            adjustable="box",
        )

        ax.grid(
            alpha=0.14,
            linewidth=0.7,
            zorder=0,
        )

        ax.tick_params(
            axis="both",
            labelsize=10.5,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)


        # -------------------------------------------------------------
        # Highlight the best model
        # -------------------------------------------------------------

        if model == best_models[method]:

            # Draw a thick rounded box around the complete panel.
            highlight_box = FancyBboxPatch(
                (
                    -0.12,
                    -0.16,
                ),
                1.24,
                1.32,
                transform=ax.transAxes,
                boxstyle="round,pad=0.018",
                facecolor="none",
                edgecolor=best_model_colour,
                linewidth=3.5,
                clip_on=False,
                zorder=10,
            )

            ax.add_patch(
                highlight_box
            )

            # Add a clear best-model badge.
            ax.text(
                0.97,
                0.04,
                "BEST MODEL",
                transform=ax.transAxes,
                ha="right",
                va="bottom",
                fontsize=11,
                fontweight="bold",
                color="#6E5200",
                bbox={
                    "boxstyle": "round,pad=0.32",
                    "facecolor": "#FFF2B2",
                    "edgecolor": best_model_colour,
                    "linewidth": 1.4,
                    "alpha": 0.96,
                },
                zorder=11,
            )


    # ---------------------------------------------------------------------
    # Add the shared legend
    # ---------------------------------------------------------------------

    legend_handles = [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="--",
            linewidth=1.3,
            label="Ideal prediction (y = x)",
        ),
        Line2D(
            [0],
            [0],
            color="#e15759",
            linestyle="-",
            linewidth=1.8,
            label="Linear fit",
        ),
        FancyBboxPatch(
            (0, 0),
            1,
            1,
            boxstyle="round,pad=0.02",
            facecolor="none",
            edgecolor=best_model_colour,
            linewidth=3.0,
            label="Lowest pooled RMSE",
        ),
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=12,
        bbox_to_anchor=(0.5, 0.018),
    )


    # ---------------------------------------------------------------------
    # Add the main figure title
    # ---------------------------------------------------------------------

    fig.suptitle(
        (
            f"Pooled out-of-fold affinity predictions: "
            f"{method.capitalize()} five-fold cross-validation"
        ),
        fontsize=20,
        y=0.985,
    )


    # ---------------------------------------------------------------------
    # Save and display the figure
    # ---------------------------------------------------------------------

    fig.savefig(
        FIGURE_ROOT
        / f"figure_predicted_vs_experimental_all_models_{method}.png",
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        FIGURE_ROOT
        / f"figure_predicted_vs_experimental_all_models_{method}.pdf",
        bbox_inches="tight",
    )

    plt.show()

## 10. Best- and worst-predicted ligands

“Best” and “worst” refer to the smallest and largest **absolute prediction errors**, not the highest and lowest predicted affinities. Tables and molecular panels are generated for every model and CV design. The two primary panels are displayed below; all fourteen are saved for appendix selection.

### Code used: Ligand error examples

I attach canonical SMILES to the compound predictions and rank absolute errors, then export molecular grids and their source rows. Lowest/highest error means prediction error, not weakest/strongest affinity.


In [ ]:
# Rank compounds by absolute prediction error after matching their molecular identity.
smiles_map = base.set_index("official_compound_group_id")["canonical_smiles"]


# Rank absolute residuals after joining SMILES by the audited compound identifier.
def error_examples(method: str, model: str, count: int = 3) -> pd.DataFrame:
    """Return the lowest- and highest-absolute-error compounds."""
    data = pd.read_csv(prediction_path(method, model), dtype={"official_compound_group_id": str})
    data["official_compound_group_id"] = data["official_compound_group_id"].str.zfill(16)
    data["canonical_smiles"] = data["official_compound_group_id"].map(smiles_map)
    data["residual"] = data["predicted_pKD"] - data["experimental_pKD"]
    data["absolute_error"] = data["residual"].abs()
    # Separate the error ranking from the compound's measured affinity.
    ordered = data.sort_values("absolute_error")
    best = ordered.head(count).assign(error_group="Lowest error")
    worst = ordered.tail(count).sort_values("absolute_error", ascending=False).assign(error_group="Highest error")
    return pd.concat([best, worst], ignore_index=True)


# Export the molecular grid together with the exact prediction rows behind it.
def save_ligand_panel(method: str, model: str) -> tuple[Path, pd.DataFrame]:
    """Save one 2×3 molecular grid and its underlying audit table."""
    examples = error_examples(method, model)
    molecules = [Chem.MolFromSmiles(smiles) for smiles in examples["canonical_smiles"]]
    legends = [
        f"{group}\nExp {truth:.2f} | Pred {prediction:.2f}\n|error| {error:.2f} | fold {fold}"
        for group, truth, prediction, error, fold in zip(
            examples["error_group"], examples["experimental_pKD"], examples["predicted_pKD"],
            examples["absolute_error"], examples["outer_fold"],
        )
    ]
    image = Draw.MolsToGridImage(
        molecules,
        molsPerRow=3,
        subImgSize=(270, 230),
        legends=legends,
        useSVG=False,
        returnPNG=True,
    )
    image_path = LIGAND_ROOT / f"{method}_{model}_lowest_highest_error.png"
    # RDKit returns bytes in a script, but its Jupyter renderer may wrap those
    # bytes in IPython.display.Image. Accept bytes, wrapped bytes and PIL images.
    # Handle RDKit's supported raster-return types without changing molecular content.
    if isinstance(image, (bytes, bytearray)):
        image_path.write_bytes(bytes(image))
    elif hasattr(image, "data") and isinstance(image.data, (bytes, bytearray)):
        image_path.write_bytes(bytes(image.data))
    elif hasattr(image, "save"):
        image.save(image_path)
    else:
        raise TypeError(f"Unsupported RDKit grid-image type: {type(image)!r}")
    examples.to_csv(TABLE_ROOT / f"table_{method}_{model}_error_examples.csv", index=False)
    return image_path, examples


panel_manifest = []
for method in ("random", "scaffold"):
    for model in MODEL_ORDER:
        image_path, examples = save_ligand_panel(method, model)
        panel_manifest.append({"CV method": method, "Model": MODEL_LABELS[model], "Panel": str(image_path), "Table rows": len(examples)})
panel_manifest = pd.DataFrame(panel_manifest)
panel_manifest.to_csv(TABLE_ROOT / "table_ligand_panel_manifest.csv", index=False)
display(panel_manifest)

### Code used: Per-fold ligand examples

I recalculate each model's compound RMSE within each outer fold, select the lowest-RMSE model, and draw its lowest- and highest-error compounds. This is a post-hoc description of test results, not validation-based selection for an unbiased new predictor.


In [ ]:
# Per-fold winners are descriptive test-result selections, not a training decision.
# -------------------------------------------------------------------------
# Prepare the compound-to-SMILES mapping
# -------------------------------------------------------------------------

smiles_source = base.copy()

smiles_source["official_compound_group_id"] = (
    smiles_source["official_compound_group_id"]
    .astype(str)
    .str.zfill(16)
)

smiles_map = (
    smiles_source
    .drop_duplicates("official_compound_group_id")
    .set_index("official_compound_group_id")[
        "canonical_smiles"
    ]
)


# -------------------------------------------------------------------------
# Font helpers
# -------------------------------------------------------------------------

# Try available fonts for consistent molecular-panel annotation sizes.
def load_panel_font(
    size: int,
    bold: bool = False,
):
    """Load a readable TrueType font."""

    if bold:
        font_path = (
            "/usr/share/fonts/truetype/dejavu/"
            "DejaVuSans-Bold.ttf"
        )
    else:
        font_path = (
            "/usr/share/fonts/truetype/dejavu/"
            "DejaVuSans.ttf"
        )

    try:
        return ImageFont.truetype(
            font_path,
            size=size,
        )
    except OSError:
        return ImageFont.load_default()


title_font = load_panel_font(
    44,
    bold=True,
)

column_heading_font = load_panel_font(
    34,
    bold=True,
)

fold_model_font = load_panel_font(
    28,
    bold=True,
)

annotation_font = load_panel_font(
    27,
    bold=False,
)

annotation_bold_font = load_panel_font(
    27,
    bold=True,
)


# -------------------------------------------------------------------------
# Centred-text helper
# -------------------------------------------------------------------------

# Measure the text width before centring it within the image tile.
def draw_centred_text(
    drawing_context,
    image_width: int,
    y_position: int,
    text: str,
    font,
    colour="black",
):
    """Draw one line of horizontally centred text."""

    text_box = drawing_context.textbbox(
        (0, 0),
        text,
        font=font,
    )

    text_width = (
        text_box[2]
        - text_box[0]
    )

    text_x = (
        image_width
        - text_width
    ) / 2

    drawing_context.text(
        (
            text_x,
            y_position - text_box[1],
        ),
        text,
        font=font,
        fill=colour,
    )


# -------------------------------------------------------------------------
# Load one model's out-of-fold predictions
# -------------------------------------------------------------------------

# Match molecular identity and calculate compound residuals from held-out predictions.
def load_model_predictions(
    method: str,
    model: str,
) -> pd.DataFrame:
    """Load predictions and calculate compound-level errors."""

    data = pd.read_csv(
        prediction_path(
            method,
            model,
        ),
        dtype={
            "official_compound_group_id": str,
        },
    )

    data["official_compound_group_id"] = (
        data["official_compound_group_id"]
        .str.zfill(16)
    )

    data["canonical_smiles"] = (
        data["official_compound_group_id"]
        .map(smiles_map)
    )

    if data["canonical_smiles"].isna().any():
        missing_ids = data.loc[
            data["canonical_smiles"].isna(),
            "official_compound_group_id",
        ].tolist()

        raise ValueError(
            "Missing canonical SMILES for compound IDs: "
            f"{missing_ids[:10]}"
        )

    data["residual"] = (
        data["predicted_pKD"]
        - data["experimental_pKD"]
    )

    data["absolute_error"] = (
        data["residual"].abs()
    )

    data["squared_error"] = (
        data["residual"] ** 2
    )

    return data


# -------------------------------------------------------------------------
# Select the lowest-RMSE model in every fold
# -------------------------------------------------------------------------

# Use test RMSE for descriptive examples only; this is not validation checkpoint selection.
def select_best_model_per_fold(
    method: str,
):
    """
    Select the model with the lowest compound-level RMSE in each outer fold.

    Returns:
    1. a fold-level model-performance table;
    2. the lowest- and highest-error compound from the selected model.
    """

    prediction_cache = {
        model: load_model_predictions(
            method,
            model,
        )
        for model in MODEL_ORDER
    }

    fold_metric_rows = []
    selected_example_rows = []

    for fold in range(5):

        # -------------------------------------------------------------
        # Calculate fold-specific RMSE for every model
        # -------------------------------------------------------------

        for model in MODEL_ORDER:

            model_data = prediction_cache[
                model
            ]

            fold_data = model_data.loc[
                model_data["outer_fold"] == fold
            ].copy()

            if fold_data.empty:
                raise ValueError(
                    f"No {method} predictions found for "
                    f"{MODEL_LABELS[model]}, fold {fold}."
                )

            fold_rmse = np.sqrt(
                np.mean(
                    (
                        fold_data["predicted_pKD"]
                        - fold_data["experimental_pKD"]
                    )
                    ** 2
                )
            )

            fold_metric_rows.append(
                {
                    "CV method": method,
                    "Fold": fold,
                    "model_key": model,
                    "Model": MODEL_LABELS[model],
                    "Compounds": len(fold_data),
                    "Fold RMSE": fold_rmse,
                }
            )

        # -------------------------------------------------------------
        # Select the model with the lowest RMSE in this fold
        # -------------------------------------------------------------

        current_fold_metrics = pd.DataFrame(
            [
                row
                for row in fold_metric_rows
                if row["Fold"] == fold
            ]
        )

        # Break equal-RMSE ties by model name for a stable descriptive selection.
        best_metric_row = (
            current_fold_metrics
            .sort_values(
                [
                    "Fold RMSE",
                    "Model",
                ]
            )
            .iloc[0]
        )

        best_model = best_metric_row[
            "model_key"
        ]

        best_model_label = best_metric_row[
            "Model"
        ]

        best_fold_rmse = float(
            best_metric_row["Fold RMSE"]
        )

        selected_model_data = (
            prediction_cache[best_model]
            .loc[
                prediction_cache[best_model][
                    "outer_fold"
                ]
                == fold
            ]
            .copy()
        )

        # Lowest absolute-error compound.
        lowest_error_row = (
            selected_model_data
            .sort_values(
                "absolute_error",
                ascending=True,
            )
            .iloc[0]
            .copy()
        )

        # Highest absolute-error compound.
        highest_error_row = (
            selected_model_data
            .sort_values(
                "absolute_error",
                ascending=False,
            )
            .iloc[0]
            .copy()
        )

        for example_type, example_row in [
            (
                "Lowest error",
                lowest_error_row,
            ),
            (
                "Highest error",
                highest_error_row,
            ),
        ]:

            result = example_row.to_dict()

            result.update(
                {
                    "CV method": method,
                    "Fold": fold,
                    "Selected model key": best_model,
                    "Selected model": best_model_label,
                    "Selected model fold RMSE": (
                        best_fold_rmse
                    ),
                    "Example type": example_type,
                }
            )

            selected_example_rows.append(
                result
            )

    return (
        pd.DataFrame(fold_metric_rows),
        pd.DataFrame(selected_example_rows),
    )


# -------------------------------------------------------------------------
# Draw one molecule
# -------------------------------------------------------------------------

# Render RDKit bond geometry into a larger image without changing the molecule.
def draw_high_resolution_molecule(
    molecule,
    width: int = 900,
    height: int = 410,
) -> Image.Image:
    """Draw one molecule without an embedded RDKit legend."""

    drawing_options = Draw.MolDrawOptions()

    drawing_options.minFontSize = 20
    drawing_options.maxFontSize = 46
    drawing_options.padding = 0.08
    drawing_options.bondLineWidth = 2.3

    image = Draw.MolToImage(
        molecule,
        size=(
            width,
            height,
        ),
        options=drawing_options,
    )

    return image.convert("RGB")


# -------------------------------------------------------------------------
# Create one best- or worst-prediction tile
# -------------------------------------------------------------------------

# Keep molecular drawing and prediction annotations in separate tile regions.
def create_fold_example_tile(
    row,
    example_type: str,
    tile_width: int = 900,
    molecule_height: int = 410,
    annotation_height: int = 245,
) -> Image.Image:
    """Create one high-resolution fold example tile."""

    molecule = Chem.MolFromSmiles(
        row["canonical_smiles"]
    )

    if molecule is None:
        raise ValueError(
            "Could not parse canonical SMILES for "
            f"{row['official_compound_group_id']}"
        )

    molecule_image = (
        draw_high_resolution_molecule(
            molecule,
            width=tile_width,
            height=molecule_height,
        )
    )

    tile_height = (
        molecule_height
        + annotation_height
    )

    tile = Image.new(
        "RGB",
        (
            tile_width,
            tile_height,
        ),
        "white",
    )

    tile.paste(
        molecule_image,
        (
            0,
            0,
        ),
    )

    drawing_context = ImageDraw.Draw(
        tile
    )

    # Select subtle highlight colours.
    if example_type == "Lowest error":
        border_colour = "#6F9278"
        annotation_background = "#F2F7F3"
    else:
        border_colour = "#AD7777"
        annotation_background = "#FAF3F3"

    # Add a pale annotation area.
    drawing_context.rectangle(
        [
            0,
            molecule_height,
            tile_width - 1,
            tile_height - 1,
        ],
        fill=annotation_background,
    )

    # Add a separator above the annotation.
    drawing_context.line(
        [
            (
                35,
                molecule_height,
            ),
            (
                tile_width - 35,
                molecule_height,
            ),
        ],
        fill=border_colour,
        width=3,
    )

    # Line 1: fold and selected model.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 18,
        (
            f"Fold {int(row['Fold'])} | "
            f"Lowest fold-RMSE model: "
            f"{row['Selected model']}"
        ),
        fold_model_font,
        colour=border_colour,
    )

    # Line 2: fold-level model RMSE.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 70,
        (
            f"Fold RMSE: "
            f"{row['Selected model fold RMSE']:.3f}"
        ),
        annotation_bold_font,
        colour="#222222",
    )

    # Line 3: experimental and predicted values.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 116,
        (
            f"Experimental pKD: "
            f"{row['experimental_pKD']:.2f}   |   "
            f"Predicted pKD: "
            f"{row['predicted_pKD']:.2f}"
        ),
        annotation_font,
        colour="#111111",
    )

    # Line 4: absolute error and compound identifier.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 162,
        (
            f"Absolute error: "
            f"{row['absolute_error']:.2f}   |   "
            f"Compound: "
            f"{row['official_compound_group_id']}"
        ),
        annotation_font,
        colour="#111111",
    )

    # Draw a subtle tile border.
    drawing_context.rectangle(
        [
            2,
            2,
            tile_width - 3,
            tile_height - 3,
        ],
        outline=border_colour,
        width=5,
    )

    return tile


# -------------------------------------------------------------------------
# Create one 5×2 panel
# -------------------------------------------------------------------------

# Arrange one low-error and one high-error example for each outer fold.
def create_fold_best_worst_panel(
    method: str,
    selected_examples: pd.DataFrame,
) -> Image.Image:
    """Create a 5-row × 2-column best/worst prediction panel."""

    tile_width = 900
    tile_height = 655

    horizontal_gap = 24
    vertical_gap = 22

    outer_margin = 25
    title_height = 105
    column_heading_height = 75

    panel_width = (
        2 * tile_width
        + horizontal_gap
        + 2 * outer_margin
    )

    panel_height = (
        title_height
        + column_heading_height
        + 5 * tile_height
        + 4 * vertical_gap
        + 2 * outer_margin
    )

    panel = Image.new(
        "RGB",
        (
            panel_width,
            panel_height,
        ),
        "white",
    )

    drawing_context = ImageDraw.Draw(
        panel
    )

    # Main title.
    draw_centred_text(
        drawing_context,
        panel_width,
        outer_margin + 8,
        (
            f"{method.capitalize()} five-fold CV: "
            "lowest fold-RMSE model in each fold"
        ),
        title_font,
        colour="#111111",
    )

    # Column headings.
    lowest_column_centre = (
        outer_margin
        + tile_width / 2
    )

    highest_column_centre = (
        outer_margin
        + tile_width
        + horizontal_gap
        + tile_width / 2
    )

    lowest_heading = (
        "Lowest-error prediction"
    )

    highest_heading = (
        "Highest-error prediction"
    )

    # Draw the left column heading.
    lowest_box = drawing_context.textbbox(
        (0, 0),
        lowest_heading,
        font=column_heading_font,
    )

    drawing_context.text(
        (
            lowest_column_centre
            - (lowest_box[2] - lowest_box[0]) / 2,
            title_height + outer_margin,
        ),
        lowest_heading,
        font=column_heading_font,
        fill="#6F9278",
    )

    # Draw the right column heading.
    highest_box = drawing_context.textbbox(
        (0, 0),
        highest_heading,
        font=column_heading_font,
    )

    drawing_context.text(
        (
            highest_column_centre
            - (highest_box[2] - highest_box[0]) / 2,
            title_height + outer_margin,
        ),
        highest_heading,
        font=column_heading_font,
        fill="#AD7777",
    )

    first_row_y = (
        outer_margin
        + title_height
        + column_heading_height
    )

    # Draw folds 0–4.
    for fold in range(5):

        fold_data = selected_examples.loc[
            selected_examples["Fold"] == fold
        ]

        lowest_row = fold_data.loc[
            fold_data["Example type"]
            == "Lowest error"
        ].iloc[0]

        highest_row = fold_data.loc[
            fold_data["Example type"]
            == "Highest error"
        ].iloc[0]

        lowest_tile = create_fold_example_tile(
            lowest_row,
            example_type="Lowest error",
        )

        highest_tile = create_fold_example_tile(
            highest_row,
            example_type="Highest error",
        )

        current_y = (
            first_row_y
            + fold
            * (
                tile_height
                + vertical_gap
            )
        )

        panel.paste(
            lowest_tile,
            (
                outer_margin,
                current_y,
            ),
        )

        panel.paste(
            highest_tile,
            (
                outer_margin
                + tile_width
                + horizontal_gap,
                current_y,
            ),
        )

    return panel


# -------------------------------------------------------------------------
# Generate random and scaffold panels
# -------------------------------------------------------------------------

panel_paths = {}
all_fold_metrics = []
all_selected_examples = []

for method in (
    "random",
    "scaffold",
):

    fold_metrics, selected_examples = (
        select_best_model_per_fold(
            method
        )
    )

    all_fold_metrics.append(
        fold_metrics
    )

    all_selected_examples.append(
        selected_examples
    )

    panel = create_fold_best_worst_panel(
        method,
        selected_examples,
    )

    panel_path = (
        LIGAND_ROOT
        / (
            f"{method}_best_model_per_fold_"
            "lowest_highest_error_5x2.png"
        )
    )

    panel.save(
        panel_path,
        format="PNG",
        dpi=(300, 300),
    )

    panel_paths[method] = panel_path


# -------------------------------------------------------------------------
# Save the numerical audit tables
# -------------------------------------------------------------------------

# Retain all candidate fold scores as an audit trail for the displayed examples.
fold_model_metrics_table = pd.concat(
    all_fold_metrics,
    ignore_index=True,
)

selected_fold_examples_table = pd.concat(
    all_selected_examples,
    ignore_index=True,
)

fold_model_metrics_table.to_csv(
    TABLE_ROOT
    / "table_model_rmse_by_outer_fold.csv",
    index=False,
)

selected_fold_examples_table.to_csv(
    TABLE_ROOT
    / "table_best_model_per_fold_error_examples.csv",
    index=False,
)

display(
    selected_fold_examples_table[
        [
            "CV method",
            "Fold",
            "Selected model",
            "Selected model fold RMSE",
            "Example type",
            "experimental_pKD",
            "predicted_pKD",
            "absolute_error",
        ]
    ].round(4)
)


# -------------------------------------------------------------------------
# Display the two panels
# -------------------------------------------------------------------------

for method in (
    "random",
    "scaffold",
):

    image = plt.imread(
        panel_paths[method]
    )

    fig, ax = plt.subplots(
        figsize=(16, 25),
    )

    ax.imshow(
        image,
        interpolation="none",
    )

    ax.axis("off")

    fig.tight_layout()

    # Also save a copy in the main figure directory.
    fig.savefig(
        FIGURE_ROOT
        / (
            f"figure_{method}_best_model_per_fold_"
            "lowest_highest_error_5x2.png"
        ),
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        FIGURE_ROOT
        / (
            f"figure_{method}_best_model_per_fold_"
            "lowest_highest_error_5x2.pdf"
        ),
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

### Code used: High-resolution ligand panels

I render the selected low- and high-error compounds as larger RDKit/Pillow tiles, keeping affinity annotations below the molecular drawings. The image helpers change presentation only.


In [ ]:
# Assemble higher-resolution panels without changing the selected compounds or errors.
# -------------------------------------------------------------------------
# Load fonts
# -------------------------------------------------------------------------

# Use the same font selection for all high-resolution ligand tiles.
def load_figure_font(
    size: int,
    bold: bool = False,
):
    """Load a high-quality font, with a fallback to PIL's default."""

    if bold:
        font_path = (
            "/usr/share/fonts/truetype/dejavu/"
            "DejaVuSans-Bold.ttf"
        )
    else:
        font_path = (
            "/usr/share/fonts/truetype/dejavu/"
            "DejaVuSans.ttf"
        )

    try:
        return ImageFont.truetype(
            font_path,
            size=size,
        )
    except OSError:
        return ImageFont.load_default()


# Large fixed fonts for the annotations.
rank_font = load_figure_font(
    size=36,
    bold=True,
)

annotation_font = load_figure_font(
    size=31,
    bold=False,
)

annotation_bold_font = load_figure_font(
    size=31,
    bold=True,
)

heading_font = load_figure_font(
    size=34,
    bold=True,
)


# -------------------------------------------------------------------------
# Draw centred PIL text
# -------------------------------------------------------------------------

# Measure the text width before centring it within the image tile.
def draw_centred_text(
    drawing_context,
    image_width: int,
    y_position: int,
    text: str,
    font,
    colour="black",
):
    """Draw one line of text centred horizontally."""

    text_box = drawing_context.textbbox(
        (0, 0),
        text,
        font=font,
    )

    text_width = (
        text_box[2]
        - text_box[0]
    )

    text_x = (
        image_width
        - text_width
    ) / 2

    drawing_context.text(
        (
            text_x,
            y_position - text_box[1],
        ),
        text,
        font=font,
        fill=colour,
    )


# -------------------------------------------------------------------------
# Draw one molecule without an RDKit legend
# -------------------------------------------------------------------------

# Convert the RDKit drawing into a Pillow image for panel composition.
def draw_molecule_image(
    molecule,
    width: int = 760,
    height: int = 430,
) -> Image.Image:
    """Draw one molecule at high resolution without embedded text."""

    drawing_options = Draw.MolDrawOptions()

    # Increase atom-label and bond readability.
    drawing_options.minFontSize = 20
    drawing_options.maxFontSize = 44
    drawing_options.padding = 0.08
    drawing_options.bondLineWidth = 2.2

    molecule_image = Draw.MolToImage(
        molecule,
        size=(
            width,
            height,
        ),
        options=drawing_options,
    )

    return molecule_image.convert("RGB")


# -------------------------------------------------------------------------
# Create one high-resolution ligand tile
# -------------------------------------------------------------------------

# Attach the compound's error annotation without drawing over its chemical structure.
def create_ligand_tile(
    row,
    rank: int,
    tile_width: int = 760,
    molecule_height: int = 430,
    annotation_height: int = 210,
) -> Image.Image:
    """Create one molecule tile with large, separately rendered annotations."""

    molecule = Chem.MolFromSmiles(
        row["canonical_smiles"]
    )

    if molecule is None:
        raise ValueError(
            "Could not parse canonical SMILES for "
            f"{row['official_compound_group_id']}"
        )

    molecule_image = draw_molecule_image(
        molecule,
        width=tile_width,
        height=molecule_height,
    )

    tile_height = (
        molecule_height
        + annotation_height
    )

    tile = Image.new(
        "RGB",
        (
            tile_width,
            tile_height,
        ),
        "white",
    )

    tile.paste(
        molecule_image,
        (
            0,
            0,
        ),
    )

    drawing_context = ImageDraw.Draw(
        tile
    )

    # Separate the structure from the numerical annotation.
    drawing_context.line(
        [
            (
                45,
                molecule_height,
            ),
            (
                tile_width - 45,
                molecule_height,
            ),
        ],
        fill="#D6D6D6",
        width=2,
    )

    # First annotation line: rank.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 18,
        f"Rank {rank}",
        rank_font,
        colour="#222222",
    )

    # Second line: observed and predicted values.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 72,
        (
            f"Experimental pKD: "
            f"{row['experimental_pKD']:.2f}   |   "
            f"Predicted pKD: "
            f"{row['predicted_pKD']:.2f}"
        ),
        annotation_font,
        colour="#111111",
    )

    # Third line: error and fold.
    draw_centred_text(
        drawing_context,
        tile_width,
        molecule_height + 123,
        (
            f"Absolute error: "
            f"{row['absolute_error']:.2f}   |   "
            f"Fold: {int(row['outer_fold'])}"
        ),
        annotation_bold_font,
        colour="#111111",
    )

    # Add a subtle border around each individual tile.
    drawing_context.rectangle(
        [
            1,
            1,
            tile_width - 2,
            tile_height - 2,
        ],
        outline="#DDDDDD",
        width=2,
    )

    return tile


# -------------------------------------------------------------------------
# Compose three ligand tiles into one row
# -------------------------------------------------------------------------

# Join equally sized molecular tiles into a consistently aligned row.
def create_ligand_row(
    row_data,
) -> Image.Image:
    """Create one row containing three high-resolution ligand tiles."""

    tiles = []

    for rank, (_, row) in enumerate(
        row_data.iterrows(),
        start=1,
    ):
        tiles.append(
            create_ligand_tile(
                row,
                rank,
            )
        )

    tile_gap = 18

    row_width = (
        sum(tile.width for tile in tiles)
        + tile_gap * (len(tiles) - 1)
    )

    row_height = max(
        tile.height
        for tile in tiles
    )

    molecular_row = Image.new(
        "RGB",
        (
            row_width,
            row_height,
        ),
        "white",
    )

    x_position = 0

    for tile in tiles:

        molecular_row.paste(
            tile,
            (
                x_position,
                0,
            ),
        )

        x_position += (
            tile.width
            + tile_gap
        )

    return molecular_row


# -------------------------------------------------------------------------
# Add a subtle row heading and border
# -------------------------------------------------------------------------

# Frame the selected error group without changing its membership.
def add_highlighted_row_frame(
    molecular_row: Image.Image,
    heading: str,
    border_colour: str,
    heading_background: str,
) -> Image.Image:
    """Add a subtle heading and thin border around a molecular row."""

    border_width = 5
    heading_height = 82

    output_width = (
        molecular_row.width
        + 2 * border_width
    )

    output_height = (
        molecular_row.height
        + heading_height
        + 2 * border_width
    )

    framed_image = Image.new(
        "RGB",
        (
            output_width,
            output_height,
        ),
        "white",
    )

    drawing_context = ImageDraw.Draw(
        framed_image
    )

    # Pale heading background.
    drawing_context.rectangle(
        [
            border_width,
            border_width,
            output_width - border_width - 1,
            border_width + heading_height,
        ],
        fill=heading_background,
    )

    # Thin muted border.
    drawing_context.rectangle(
        [
            1,
            1,
            output_width - 2,
            output_height - 2,
        ],
        outline=border_colour,
        width=border_width,
    )

    draw_centred_text(
        drawing_context,
        output_width,
        border_width + 20,
        heading,
        heading_font,
        colour=border_colour,
    )

    framed_image.paste(
        molecular_row,
        (
            border_width,
            border_width + heading_height,
        ),
    )

    return framed_image


# -------------------------------------------------------------------------
# Generate one lowest/highest-error panel
# -------------------------------------------------------------------------

# Use the same ranked prediction rows for the image and its audit CSV.
def generate_high_resolution_error_panel(
    method: str,
    model: str,
    count: int = 3,
):
    """Generate three lowest-error and three highest-error compounds."""

    examples = error_examples(
        method,
        model,
        count=count,
    )

    lowest_error = (
        examples.loc[
            examples["error_group"] == "Lowest error"
        ]
        .sort_values(
            "absolute_error",
            ascending=True,
        )
        .head(count)
        .reset_index(drop=True)
    )

    highest_error = (
        examples.loc[
            examples["error_group"] == "Highest error"
        ]
        .sort_values(
            "absolute_error",
            ascending=False,
        )
        .head(count)
        .reset_index(drop=True)
    )

    if len(lowest_error) != count:
        raise ValueError(
            f"Expected {count} lowest-error examples, "
            f"but found {len(lowest_error)}."
        )

    if len(highest_error) != count:
        raise ValueError(
            f"Expected {count} highest-error examples, "
            f"but found {len(highest_error)}."
        )

    # Draw each row without RDKit legends.
    lowest_row = create_ligand_row(
        lowest_error
    )

    highest_row = create_ligand_row(
        highest_error
    )

    # Add subtle green and red framing.
    lowest_row = add_highlighted_row_frame(
        lowest_row,
        heading="Three lowest-error predictions",
        border_colour="#6F9278",
        heading_background="#F2F7F3",
    )

    highest_row = add_highlighted_row_frame(
        highest_row,
        heading="Three highest-error predictions",
        border_colour="#AD7777",
        heading_background="#FAF3F3",
    )

    row_gap = 30

    combined_width = max(
        lowest_row.width,
        highest_row.width,
    )

    combined_height = (
        lowest_row.height
        + highest_row.height
        + row_gap
    )

    combined_image = Image.new(
        "RGB",
        (
            combined_width,
            combined_height,
        ),
        "white",
    )

    lowest_x = (
        combined_width
        - lowest_row.width
    ) // 2

    highest_x = (
        combined_width
        - highest_row.width
    ) // 2

    combined_image.paste(
        lowest_row,
        (
            lowest_x,
            0,
        ),
    )

    combined_image.paste(
        highest_row,
        (
            highest_x,
            lowest_row.height + row_gap,
        ),
    )

    # Save the numerical examples.
    examples.to_csv(
        TABLE_ROOT
        / (
            f"table_{method}_{model}_"
            "error_examples_high_resolution.csv"
        ),
        index=False,
    )

    output_path = (
        LIGAND_ROOT
        / (
            f"{method}_{model}_"
            "lowest_highest_error_high_resolution.png"
        )
    )

    combined_image.save(
        output_path,
        format="PNG",
        dpi=(300, 300),
    )

    return output_path


# -------------------------------------------------------------------------
# Generate the random and scaffold panels
# -------------------------------------------------------------------------

high_resolution_panels = {}

for method in (
    "random",
    "scaffold",
):

    model = best_models[method]

    high_resolution_panels[method] = (
        generate_high_resolution_error_panel(
            method,
            model,
            count=3,
        )
    )


# -------------------------------------------------------------------------
# Assemble the final dissertation figure
# -------------------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    1,
    figsize=(22, 18),
)

for ax, method in zip(
    axes,
    ("random", "scaffold"),
):

    model = best_models[method]

    image = plt.imread(
        high_resolution_panels[method]
    )

    # Do not smooth the rasterised annotations.
    ax.imshow(
        image,
        interpolation="none",
    )

    ax.set_title(
        (
            f"{method.capitalize()} five-fold CV: "
            f"{MODEL_LABELS[model]}"
        ),
        fontsize=20,
        fontweight="semibold",
        pad=18,
    )

    ax.axis("off")

fig.suptitle(
    "Lowest- and highest-error out-of-fold ligand predictions",
    fontsize=23,
    y=0.99,
)

fig.tight_layout(
    rect=(
        0.01,
        0.01,
        0.99,
        0.96,
    ),
    h_pad=3.0,
)

fig.savefig(
    FIGURE_ROOT
    / "figure_error_ligands_best_models.png",
    dpi=400,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_ROOT
    / "figure_error_ligands_best_models.pdf",
    dpi=400,
    bbox_inches="tight",
)

plt.show()

## 11. Contextual OpenBind benchmark comparison

The official values are zero-shot or untrained-reference results on a 494-compound benchmark cohort. Project models were trained on the 474-compound curated OpenBind cohort. The table is therefore **contextual and not a direct leaderboard comparison**.

### Code used: External-reference table

I read the official benchmark table when it is available; otherwise the cell uses the explicitly stored reference values. I label reference and project rows separately because their cohorts and evaluation protocols differ.


In [ ]:
# Keep official reference values separate from target-trained CV results.
# -------------------------------------------------------------------------
# Load the released OpenBind benchmark results
# -------------------------------------------------------------------------

benchmark_path = (
    ROOT.parent
    / "EV-A71_2A_benchmark"
    / "plotting"
    / "tables"
    / "affinity_metrics.csv"
)

if benchmark_path.is_file():

    benchmark = (
        pd.read_csv(benchmark_path)
        .rename(
            columns={
                "method": "Method",
                "n": "Compounds",
                "Spearman rho": "Spearman",
            }
        )
    )

# These fallback rows are stored reference numbers, not newly evaluated predictions.
else:

    # Fallback values used if the benchmark table is unavailable.
    benchmark = pd.DataFrame(
        [
            [
                "molecular_weight",
                494,
                np.nan,
                0.483,
            ],
            [
                "clogp",
                494,
                np.nan,
                0.174,
            ],
            [
                "aev-plig",
                494,
                1.090,
                0.227,
            ],
            [
                "gnina",
                494,
                1.528,
                0.453,
            ],
            [
                "smina",
                494,
                1.530,
                0.255,
            ],
            [
                "aqaffinity",
                494,
                1.632,
                0.117,
            ],
            [
                "boltz-2",
                494,
                1.091,
                0.397,
            ],
        ],
        columns=[
            "Method",
            "Compounds",
            "RMSE",
            "Spearman",
        ],
    )


# -------------------------------------------------------------------------
# Define the graph-based project models
# -------------------------------------------------------------------------

# Exclude only the Morgan fingerprint MLP.
# The 2D GNN is retained because it is a learned graph model.
project_gnn_models = [
    model
    for model in MODEL_ORDER
    if model != "morgan_mlp"
]


# -------------------------------------------------------------------------
# Add every GNN-based project result
# -------------------------------------------------------------------------

project_context = []

for method in (
    "random",
    "scaffold",
):

    for model in project_gnn_models:

        metric = cv_summaries[
            method
        ][
            "pooled_out_of_fold_metrics"
        ][
            model
        ]

        project_context.append(
            {
                "Method": (
                    f"{method.capitalize()} CV: "
                    f"{MODEL_LABELS[model]}"
                ),
                "Compounds": 474,
                "RMSE": metric["rmse"],
                "Spearman": metric["spearman_r"],
                "Comparison class": "OpenBind-trained CV",
                "CV design": method,
                "Model key": model,
            }
        )


# -------------------------------------------------------------------------
# Add context columns to the released benchmark rows
# -------------------------------------------------------------------------

benchmark = benchmark.copy()

benchmark[
    "Comparison class"
] = "Released zero-shot/reference"

benchmark[
    "CV design"
] = "Not applicable"

benchmark[
    "Model key"
] = pd.NA


# -------------------------------------------------------------------------
# Combine benchmark and project results
# -------------------------------------------------------------------------

benchmark_context = pd.concat(
    [
        benchmark,
        pd.DataFrame(project_context),
    ],
    ignore_index=True,
    sort=False,
)


# -------------------------------------------------------------------------
# Save and display the contextual comparison
# -------------------------------------------------------------------------

benchmark_context.to_csv(
    TABLE_ROOT
    / "table_openbind_benchmark_context.csv",
    index=False,
)

display(
    benchmark_context.round(4)
)

### Code used: External-reference plot

I select the three unmasked 3D project models for the contextual bar chart and retain the source cohort labels. These rows are displayed together, not pooled into a common evaluation.


In [ ]:
# Use colour and cohort labels to distinguish evaluation protocols.
# -------------------------------------------------------------------------
# Select the project models to include.
# -------------------------------------------------------------------------

selected_project_models = [
    "3d_gnn",
    "3d_alignn",
    "adapted_mgt",
]


# -------------------------------------------------------------------------
# Extract only the released benchmark rows.
# -------------------------------------------------------------------------

# benchmark_context may contain both benchmark and project results.
# Retain only the released zero-shot/reference results here.
if "Comparison class" in benchmark_context.columns:
    benchmark_rows = benchmark_context[
        benchmark_context["Comparison class"]
        == "Released zero-shot/reference"
    ].copy()
else:
    # Fall back to the original benchmark table if the comparison-class
    # column has not yet been created.
    benchmark_rows = benchmark.copy()
    benchmark_rows["Comparison class"] = (
        "Released zero-shot/reference"
    )

# Ensure benchmark rows have the required columns.
benchmark_rows["CV design"] = "Released benchmark"
benchmark_rows["Model key"] = pd.NA

# Remove released methods for which RMSE was not reported.
benchmark_rows = benchmark_rows.dropna(
    subset=["RMSE"]
).copy()

# Use the benchmark method name as its plot label.
benchmark_rows["Plot label"] = benchmark_rows["Method"]


# -------------------------------------------------------------------------
# Reconstruct the selected project results directly from CV summaries.
# -------------------------------------------------------------------------

project_rows = []

for method in ("random", "scaffold"):
    for model_key in selected_project_models:

        # Read the pooled out-of-fold metrics for this model.
        metrics = (
            cv_summaries[method]
            ["pooled_out_of_fold_metrics"]
            [model_key]
        )

        # Add one row for this model and CV design.
        project_rows.append(
            {
                "Method": MODEL_LABELS[model_key],
                "Plot label": (
                    f"{MODEL_LABELS[model_key]} — "
                    f"{method.capitalize()} CV"
                ),
                "Model key": model_key,
                "CV design": method,
                "Compounds": 474,
                "RMSE": metrics["rmse"],
                "Spearman": metrics["spearman_r"],
                "Comparison class": "OpenBind-trained CV",
            }
        )

project_rows = pd.DataFrame(project_rows)


# -------------------------------------------------------------------------
# Sort the project models consistently.
# -------------------------------------------------------------------------

cv_order = {
    "random": 0,
    "scaffold": 1,
}

model_order = {
    "3d_gnn": 0,
    "3d_alignn": 1,
    "adapted_mgt": 2,
}

project_rows["CV order"] = (
    project_rows["CV design"]
    .map(cv_order)
)

project_rows["Model order"] = (
    project_rows["Model key"]
    .map(model_order)
)

project_rows = project_rows.sort_values(
    ["CV order", "Model order"]
).reset_index(drop=True)


# -------------------------------------------------------------------------
# Combine benchmark and project results.
# -------------------------------------------------------------------------

required_columns = [
    "Method",
    "Plot label",
    "Model key",
    "CV design",
    "Compounds",
    "RMSE",
    "Spearman",
    "Comparison class",
]

# Add any absent optional columns to benchmark_rows.
for column in required_columns:
    if column not in benchmark_rows.columns:
        benchmark_rows[column] = pd.NA

rmse_plot = pd.concat(
    [
        benchmark_rows[required_columns],
        project_rows[required_columns],
    ],
    ignore_index=True,
)


# -------------------------------------------------------------------------
# Assign colours according to evaluation design.
# -------------------------------------------------------------------------

# Encode evaluation source and CV design, not the numerical rank, in bar colours.
def result_colour(row):
    """Return the plotting colour for one result."""

    if (
        row["Comparison class"]
        == "Released zero-shot/reference"
    ):
        return "#9d9d9d"

    if row["CV design"] == "random":
        return "#4c78a8"

    if row["CV design"] == "scaffold":
        return "#f28e2b"

    return "#cccccc"


bar_colours = [
    result_colour(row)
    for _, row in rmse_plot.iterrows()
]


# -------------------------------------------------------------------------
# Create the horizontal bar chart.
# -------------------------------------------------------------------------

figure_height = max(
    6.5,
    0.52 * len(rmse_plot),
)

fig, ax = plt.subplots(
    figsize=(11, figure_height)
)

bars = ax.barh(
    rmse_plot["Plot label"],
    rmse_plot["RMSE"],
    color=bar_colours,
    edgecolor="white",
    linewidth=0.8,
    zorder=2,
)

# Show released results first at the top.
ax.invert_yaxis()


# -------------------------------------------------------------------------
# Add RMSE values to the ends of the bars.
# -------------------------------------------------------------------------

maximum_rmse = rmse_plot["RMSE"].max()
label_offset = maximum_rmse * 0.012

for bar, value in zip(
    bars,
    rmse_plot["RMSE"],
):
    ax.text(
        value + label_offset,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        ha="left",
        va="center",
        fontsize=10,
        fontweight="semibold",
    )

# Leave room for the numerical labels.
ax.set_xlim(
    0,
    maximum_rmse * 1.12,
)


# -------------------------------------------------------------------------
# Configure the axes and title.
# -------------------------------------------------------------------------

ax.set_xlabel(
    "Reported RMSE (pKD)",
    fontsize=12,
)

ax.set_ylabel(
    "Method",
    fontsize=12,
)

ax.set_title(
    "Contextual OpenBind affinity results: "
    "released benchmarks and trained 3D graph models",
    fontsize=14,
    pad=14,
)

ax.grid(
    axis="x",
    alpha=0.20,
    linewidth=0.7,
    zorder=0,
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# -------------------------------------------------------------------------
# Add the evaluation-design legend.
# -------------------------------------------------------------------------

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markerfacecolor="#9d9d9d",
        markeredgecolor="none",
        markersize=10,
        label="Released zero-shot/reference",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markerfacecolor="#4c78a8",
        markeredgecolor="none",
        markersize=10,
        label="Random five-fold CV",
    ),
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="none",
        markerfacecolor="#f28e2b",
        markeredgecolor="none",
        markersize=10,
        label="Scaffold five-fold CV",
    ),
]

ax.legend(
    handles=legend_handles,
    frameon=False,
    loc="lower right",
    fontsize=10,
)


# -------------------------------------------------------------------------
# Add the scientific-comparison qualification.
# -------------------------------------------------------------------------

fig.text(
    0.5,
    0.01,
    (
        "Contextual comparison only: released benchmark methods and "
        "project models used different compound cohorts and evaluation protocols."
    ),
    ha="center",
    va="bottom",
    fontsize=9,
    color="#555555",
)


# -------------------------------------------------------------------------
# Save and display the figure.
# -------------------------------------------------------------------------

fig.tight_layout(
    rect=(0, 0.045, 1, 1)
)

fig.savefig(
    FIGURE_ROOT
    / "figure_openbind_benchmark_3d_models_context.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_ROOT
    / "figure_openbind_benchmark_3d_models_context.pdf",
    bbox_inches="tight",
)

plt.show()


# -------------------------------------------------------------------------
# Save and display the table underlying the plot.
# -------------------------------------------------------------------------

display_columns = [
    "Plot label",
    "Compounds",
    "RMSE",
    "Spearman",
    "Comparison class",
]

context_table = rmse_plot[
    display_columns
].copy()

context_table.to_csv(
    TABLE_ROOT
    / "table_openbind_benchmark_3d_models_context.csv",
    index=False,
)

display(
    context_table.round(4)
)

### Code used: Combined CV score tables

I reread both CV summary directories to export one fold-level table and one pooled model table. This cell resolves its own paths, so those settings must also be updated if result folders are renamed.


In [ ]:
# These standalone summary cells resolve their own result paths.
# -------------------------------------------------------------------------
# Locate the MGT project directory.
# -------------------------------------------------------------------------

# These appended tables find their own defaults rather than reusing RESULT_ROOTS.
def find_project_root() -> Path:
    """Find the MGT directory containing the completed CV summaries."""

    candidates = [Path.cwd(), *Path.cwd().parents]

    for candidate in candidates:
        random_summary = (
            candidate
            / "output"
            / "openbind_random_cv"
            / "summary"
            / "cv_summary.json"
        )

        if random_summary.is_file():
            return candidate

    raise FileNotFoundError(
        "Could not find the MGT project root containing "
        "output/openbind_random_cv/summary/cv_summary.json"
    )


PROJECT_ROOT = find_project_root()

TABLE_ROOT = (
    PROJECT_ROOT
    / "output"
    / "dissertation_analysis"
    / "tables"
)

TABLE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Define the CV result locations.
# -------------------------------------------------------------------------

# These paths point to summary directories; the next table cell expects CSV paths.
CV_PATHS = {
    "Random": (
        PROJECT_ROOT
        / "output"
        / "openbind_random_cv"
        / "summary"
    ),
    "Scaffold": (
        PROJECT_ROOT
        / "output"
        / "openbind_scaffold_cv"
        / "summary"
    ),
}


# -------------------------------------------------------------------------
# Define a consistent model order and display names.
# -------------------------------------------------------------------------

MODEL_ORDER = [
    "morgan_mlp",
    "2d_gnn",
    "3d_gnn",
    "3d_alignn",
    "adapted_mgt",
    "masked_alignn",
    "masked_mgt",
]

MODEL_LABELS = {
    "morgan_mlp": "Morgan MLP",
    "2d_gnn": "2D GNN",
    "3d_gnn": "3D distance GNN",
    "3d_alignn": "3D ALIGNN",
    "adapted_mgt": "Adapted MGT",
    "masked_alignn": "Masked ALIGNN",
    "masked_mgt": "Masked MGT",
}


# -------------------------------------------------------------------------
# Load every individual outer-fold result.
# -------------------------------------------------------------------------

fold_frames = []

for cv_design, summary_directory in CV_PATHS.items():

    fold_path = summary_directory / "fold_metrics.csv"

    if not fold_path.is_file():
        raise FileNotFoundError(
            f"Missing fold metrics file: {fold_path}"
        )

    fold_frame = pd.read_csv(fold_path)

    fold_frame["CV design"] = cv_design

    fold_frames.append(fold_frame)


all_fold_scores = pd.concat(
    fold_frames,
    ignore_index=True,
)

all_fold_scores["Model"] = all_fold_scores["model_key"].map(
    MODEL_LABELS
)

all_fold_scores["Model"] = pd.Categorical(
    all_fold_scores["Model"],
    categories=[
        MODEL_LABELS[model]
        for model in MODEL_ORDER
    ],
    ordered=True,
)

all_fold_scores["CV design"] = pd.Categorical(
    all_fold_scores["CV design"],
    categories=["Random", "Scaffold"],
    ordered=True,
)

all_fold_scores = (
    all_fold_scores[
        [
            "CV design",
            "Model",
            "outer_fold",
            "test_compounds",
            "mae",
            "rmse",
            "r2",
            "pearson_r",
            "spearman_r",
        ]
    ]
    .rename(
        columns={
            "outer_fold": "Outer fold",
            "test_compounds": "Test compounds",
            "mae": "MAE",
            "rmse": "RMSE",
            "r2": "R²",
            "pearson_r": "Pearson r",
            "spearman_r": "Spearman ρ",
        }
    )
    .sort_values(
        ["CV design", "Model", "Outer fold"]
    )
    .reset_index(drop=True)
)


# Save the complete 70-row fold table.
all_fold_scores.to_csv(
    TABLE_ROOT / "table_all_cv_fold_scores.csv",
    index=False,
)


# -------------------------------------------------------------------------
# Load pooled out-of-fold metrics and fold-level mean/SD values.
# -------------------------------------------------------------------------

summary_rows = []

for cv_design, summary_directory in CV_PATHS.items():

    summary_path = summary_directory / "cv_summary.json"

    with summary_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        summary = json.load(handle)

    pooled_metrics = summary[
        "pooled_out_of_fold_metrics"
    ]

    for model_key in MODEL_ORDER:

        metrics = pooled_metrics[model_key]

        summary_rows.append(
            {
                "CV design": cv_design,
                "Model": MODEL_LABELS[model_key],
                "Compounds": int(
                    metrics["n_compounds"]
                ),
                "MAE": metrics["mae"],
                "MSE": metrics["mse"],
                "RMSE": metrics["rmse"],
                "R²": metrics["r2"],
                "Pearson r": metrics["pearson_r"],
                "Spearman ρ": metrics["spearman_r"],
                "Fold RMSE mean": (
                    metrics["fold_mean_rmse"]
                ),
                "Fold RMSE SD": (
                    metrics["fold_sd_rmse"]
                ),
            }
        )


summary_table = pd.DataFrame(summary_rows)

summary_table["Model"] = pd.Categorical(
    summary_table["Model"],
    categories=[
        MODEL_LABELS[model]
        for model in MODEL_ORDER
    ],
    ordered=True,
)

summary_table["CV design"] = pd.Categorical(
    summary_table["CV design"],
    categories=["Random", "Scaffold"],
    ordered=True,
)

summary_table = (
    summary_table
    .sort_values(["CV design", "Model"])
    .reset_index(drop=True)
)


# Create a dissertation-friendly mean ± SD column.
summary_table["Fold RMSE mean ± SD"] = summary_table.apply(
    lambda row: (
        f"{row['Fold RMSE mean']:.4f} "
        f"± {row['Fold RMSE SD']:.4f}"
    ),
    axis=1,
)


# Mark the lowest pooled RMSE within each CV design.
summary_table["Best pooled RMSE"] = (
    summary_table["RMSE"]
    == summary_table.groupby(
        "CV design",
        observed=True,
    )["RMSE"].transform("min")
)


# Save the complete numerical table.
summary_table.to_csv(
    TABLE_ROOT / "table_all_cv_model_scores.csv",
    index=False,
)


# -------------------------------------------------------------------------
# Create the concise dissertation table.
# -------------------------------------------------------------------------

dissertation_table = summary_table[
    [
        "CV design",
        "Model",
        "Compounds",
        "MAE",
        "RMSE",
        "R²",
        "Pearson r",
        "Spearman ρ",
        "Fold RMSE mean ± SD",
    ]
].copy()


# Record the row indices of the best models before styling.
best_indices = set(
    summary_table.index[
        summary_table["Best pooled RMSE"]
    ]
)


# Highlight the lowest pooled RMSE within each CV design only in the displayed table.
def highlight_best_model(row):
    """Highlight the lowest pooled-RMSE model for each CV design."""

    if row.name in best_indices:
        return [
            "background-color: #e2f0d9; "
            "font-weight: bold"
        ] * len(row)

    return [""] * len(row)


styled_table = (
    dissertation_table.style
    .format(
        {
            "MAE": "{:.4f}",
            "RMSE": "{:.4f}",
            "R²": "{:.4f}",
            "Pearson r": "{:.4f}",
            "Spearman ρ": "{:.4f}",
        }
    )
    .apply(
        highlight_best_model,
        axis=1,
    )
    .set_caption(
        "Compound-level five-fold cross-validation performance"
    )
)


display(styled_table)


# -------------------------------------------------------------------------
# Display the complete individual-fold table.
# -------------------------------------------------------------------------

display(
    all_fold_scores.style
    .format(
        {
            "MAE": "{:.4f}",
            "RMSE": "{:.4f}",
            "R²": "{:.4f}",
            "Pearson r": "{:.4f}",
            "Spearman ρ": "{:.4f}",
        }
    )
    .set_caption(
        "Individual outer-fold scores for every model and CV design"
    )
)


print(
    "Saved:",
    TABLE_ROOT / "table_all_cv_model_scores.csv",
)

print(
    "Saved:",
    TABLE_ROOT / "table_all_cv_fold_scores.csv",
)

### Code used: Best model in each outer fold

I group the saved fold metrics by outer fold and retain the row with minimum test RMSE for each. The resulting random and scaffold tables are descriptive test-set rankings, not a validation-based selection rule.


In [ ]:
# Minimum outer-test RMSE is used here only to describe each fold's observed winner.
# -------------------------------------------------------------------------
# Locate the MGT project root.
# -------------------------------------------------------------------------

# These appended tables find their own defaults rather than reusing RESULT_ROOTS.
def find_project_root() -> Path:
    """Locate the project directory containing the CV output folders."""

    for candidate in [Path.cwd(), *Path.cwd().parents]:

        random_metrics = (
            candidate
            / "output"
            / "openbind_random_cv"
            / "summary"
            / "fold_metrics.csv"
        )

        if random_metrics.is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate "
        "output/openbind_random_cv/summary/fold_metrics.csv"
    )


PROJECT_ROOT = find_project_root()

TABLE_ROOT = (
    PROJECT_ROOT
    / "output"
    / "dissertation_analysis"
    / "tables"
)

TABLE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Define the two completed cross-validation experiments.
# -------------------------------------------------------------------------

# Here each path points directly to fold_metrics.csv.
CV_PATHS = {
    "Random": (
        PROJECT_ROOT
        / "output"
        / "openbind_random_cv"
        / "summary"
        / "fold_metrics.csv"
    ),
    "Scaffold": (
        PROJECT_ROOT
        / "output"
        / "openbind_scaffold_cv"
        / "summary"
        / "fold_metrics.csv"
    ),
}


# Standardise model labels used in the dissertation.
MODEL_LABELS = {
    "morgan_mlp": "Morgan MLP",
    "2d_gnn": "2D GNN",
    "3d_gnn": "3D distance GNN",
    "3d_alignn": "3D ALIGNN",
    "adapted_mgt": "Adapted MGT",
    "masked_alignn": "Masked ALIGNN",
    "masked_mgt": "Masked MGT",
}


# -------------------------------------------------------------------------
# Select the lowest-RMSE model independently within every outer fold.
# -------------------------------------------------------------------------

# Retain the complete metric row for the minimum-RMSE model in each outer test fold.
def create_best_fold_table(
    cv_design: str,
    metrics_path: Path,
) -> pd.DataFrame:
    """
    Return the model with the lowest compound-level RMSE in each fold.

    Models within a fold are evaluated using the same held-out compounds,
    making the within-fold ranking a matched comparison.
    """

    if not metrics_path.is_file():
        raise FileNotFoundError(
            f"Missing CV metrics file: {metrics_path}"
        )

    metrics = pd.read_csv(metrics_path)

    required_columns = {
        "model_key",
        "outer_fold",
        "test_compounds",
        "mae",
        "rmse",
        "r2",
        "pearson_r",
        "spearman_r",
    }

    missing_columns = required_columns.difference(
        metrics.columns
    )

    if missing_columns:
        raise KeyError(
            f"{metrics_path} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    # Obtain the row index containing the lowest RMSE in each fold.
    # Choose one observed test-result row per fold; no model is retrained.
    best_indices = (
        metrics
        .groupby(
            "outer_fold",
            observed=True,
        )["rmse"]
        .idxmin()
    )

    # Extract and order the winning rows.
    best = (
        metrics
        .loc[best_indices]
        .sort_values("outer_fold")
        .copy()
    )

    best["Model"] = best["model_key"].map(
        MODEL_LABELS
    )

    best["CV design"] = cv_design

    best = (
        best[
            [
                "outer_fold",
                "Model",
                "test_compounds",
                "mae",
                "rmse",
                "r2",
                "pearson_r",
                "spearman_r",
            ]
        ]
        .rename(
            columns={
                "outer_fold": "Outer fold",
                "test_compounds": "Test compounds",
                "mae": "MAE",
                "rmse": "RMSE",
                "r2": "R²",
                "pearson_r": "Pearson r",
                "spearman_r": "Spearman ρ",
            }
        )
        .reset_index(drop=True)
    )

    return best


random_best_models = create_best_fold_table(
    cv_design="Random",
    metrics_path=CV_PATHS["Random"],
)

scaffold_best_models = create_best_fold_table(
    cv_design="Scaffold",
    metrics_path=CV_PATHS["Scaffold"],
)


# -------------------------------------------------------------------------
# Save both tables.
# -------------------------------------------------------------------------

random_best_models.to_csv(
    TABLE_ROOT / "table_random_cv_best_model_per_fold.csv",
    index=False,
)

scaffold_best_models.to_csv(
    TABLE_ROOT / "table_scaffold_cv_best_model_per_fold.csv",
    index=False,
)


# -------------------------------------------------------------------------
# Format and display the tables.
# -------------------------------------------------------------------------

metric_format = {
    "MAE": "{:.4f}",
    "RMSE": "{:.4f}",
    "R²": "{:.4f}",
    "Pearson r": "{:.4f}",
    "Spearman ρ": "{:.4f}",
}


# Emphasise the winning-model columns without modifying saved metric values.
def highlight_winner_columns(column):
    """Highlight the model name and winning RMSE."""

    if column.name in {"Model", "RMSE"}:
        return [
            "background-color: #e2f0d9; font-weight: bold;"
        ] * len(column)

    return [""] * len(column)


random_styled = (
    random_best_models.style
    .format(metric_format)
    .apply(
        highlight_winner_columns,
        axis=0,
    )
    .set_caption(
        "Best-performing model in each random-CV outer fold"
    )
    .set_properties(
        **{
            "text-align": "center",
        }
    )
)

scaffold_styled = (
    scaffold_best_models.style
    .format(metric_format)
    .apply(
        highlight_winner_columns,
        axis=0,
    )
    .set_caption(
        "Best-performing model in each scaffold-CV outer fold"
    )
    .set_properties(
        **{
            "text-align": "center",
        }
    )
)


display(random_styled)
display(scaffold_styled)


print(
    "Saved:",
    TABLE_ROOT / "table_random_cv_best_model_per_fold.csv",
)

print(
    "Saved:",
    TABLE_ROOT / "table_scaffold_cv_best_model_per_fold.csv",
)

## 12. Hypothesis summary and dissertation reporting table

The outcome labels below are descriptive conclusions from the paired model hierarchy. They should be interpreted alongside the bootstrap intervals rather than from model rank alone.

### Code used: Reporting summary table

I export the hypothesis labels and outcome descriptions written explicitly in this cell. They are a reporting summary, not conclusions generated automatically from a statistical test.


In [ ]:
# These outcome labels are entered for reporting, not inferred by this cell.
# Review these fixed descriptions when analysing a different experiment.
hypothesis_table = pd.DataFrame([
    ["H1", "Learned 2D graphs outperform fixed Morgan fingerprints", "Modestly supported"],
    ["H2", "Crystallographic distances improve upon the 2D GNN", "Supported"],
    ["H3", "Angles and MGT complexity further improve performance", "Not consistently supported"],
    ["H4", "Masked atom-feature pretraining improves generalisation", "Not consistently supported"],
    ["H5", "Scaffold CV is substantially harder than random CV", "Not clearly supported"],
], columns=["Hypothesis", "Statement", "Outcome"])
hypothesis_table.to_csv(TABLE_ROOT / "table_hypothesis_outcomes.csv", index=False)
display(hypothesis_table)

## 13. Output audit and session information

The final cell lists every generated artifact and records package versions. Copy figures into the dissertation only after checking their captions, dimensions and labels at the final page size.

### Code used: Output and session audit

I record the runtime versions and list the saved figure/table files with their sizes. This makes the exports traceable; file presence alone does not verify their scientific interpretation.


In [ ]:
# Record the session and exported files after all analysis cells have run.
session_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "rdkit": rdBase.rdkitVersion,
    "bootstrap_iterations": N_BOOTSTRAP,
    "seed": RNG_SEED,
    "dataset_sha256": dataset_metadata["sha256"]["curated_structures"],
}
(ANALYSIS_ROOT / "analysis_session.json").write_text(json.dumps(session_info, indent=2) + "\n", encoding="utf-8")

artifact_rows = []
for kind, directory in [("Figure", FIGURE_ROOT), ("Table", TABLE_ROOT)]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            artifact_rows.append({"Type": kind, "File": str(path.relative_to(ROOT)), "Bytes": path.stat().st_size})
artifact_manifest = pd.DataFrame(artifact_rows)
artifact_manifest.to_csv(ANALYSIS_ROOT / "artifact_manifest.csv", index=False)

print(json.dumps(session_info, indent=2))
print(f"\nGenerated {sum(artifact_manifest['Type'] == 'Figure')} figure files and {sum(artifact_manifest['Type'] == 'Table')} table files.")
display(artifact_manifest)

## Interpretation checklist

- Treat pooled out-of-fold metrics as the primary performance estimates.
- Use fold SD to describe variability, not as a confidence interval.
- Use paired bootstrap intervals before describing small RMSE differences as reliable.
- Do not describe the contextual OpenBind benchmark table as a direct comparison.
- Do not infer causal structural alerts from the five best/worst ligand examples.
- The ligand-only adapted MGT cannot model direct protein–ligand contacts.
- Keep exhaustive learning curves and all fourteen ligand panels in the appendix; use the selected summary figures in the main chapter.